# Setup
Run all cells in this section before anything else.

In [ ]:
# DEPENDENCIES

import subprocess
import os
import pandas as pd
from pathlib import Path
import time
import re
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import h5py
import glob
import json
import shutil
from bioio import BioImage

In [ ]:
# CONFIGURATION

# USER PATHS
BASE_DIR = r"path/to/base/directory"
ILASTIK_EXE = r"path/to/ilastik.exe"
FIJI_PATH = r"path/to/Fiji.app"

# predict directory (change for each protein folder, re-run Step 3 + 4 + ImageJ analysis)
PREDICT_DIR = r"path/to/protein/folder"

# OUTPUT
OUTPUT_DIR = os.path.join(PREDICT_DIR, "imagej_results")

# DERIVED PATHS
INPUT_DIR = PREDICT_DIR  # need for compatibility
TRAINING_DIR = os.path.join(BASE_DIR, "vessel_classifier")
ILP_PROJECT = os.path.join(TRAINING_DIR, "vessel_pixel_classification_3D.ilp")
os.makedirs(TRAINING_DIR, exist_ok=True)

# protein subfolders (for Step 1 export)
PROTEINS = ['CD44', 'SDC1', 'WGA', 'HABP', 'ZO-1', 'eNOS', 'ICAM-1']

# cohort and brain region categories
MOUSE_TYPES = ['YOUNG', 'TBI', 'OLD', 'AD']
BRAIN_REGIONS = ['PFC', 'HIPP', 'MID']

# target filters (which categories to process in pipeline)
TARGET_MOUSE_TYPES = ['YOUNG', 'TBI', 'OLD', 'AD']
TARGET_REGIONS = ['PFC', 'HIPP', 'MID']

# ilastik training files (backed up before Step 1)
TRAINING_H5S = [
    # list h5 filenames used to train the ilastik project
    # these are protected from overwrite during RE_EXPORT
]

# ilastik output channels (3-class: vessel, microglia, background)
VESSEL_CHANNEL = 0
MICROGLIA_CHANNEL = 1

# ilastik prediction control
REANALYZE = False  # True = re-predict all files after retraining
RE_EXPORT = False  # True = re-export all h5s (use if channel detection was wrong)

# ilastik TL integration mode (how ImageJ decides to use ilastik output)
# 'auto'   = AD/TBI/OLD use dominant-class automatically; YOUNG uses raw TL
# 'manual' = show all options in FIJI (original, binary, prob, dominant-class) and let user choose
# 'off'    = never use ilastik, always raw TL
ILASTIK_TL_MODE = 'manual'

# ImageJ analysis parameters
ROLLING_BALL_RADIUS = 50

TL_GAUSSIAN_BLUR = 0
TL_CLOSE_ITERATIONS = 1
TL_FILL_HOLES = True
TL_REMOVE_SMALL = 5

DEBUG_MODE = False
USE_ROI = False
SHOW_BACKGROUND_UI = False
AUTO_SAVE_IMAGES = False
SAVE_VESSEL_OUTLINE = True   # draw vessel mask outline on saved overlay images

# saved overlay image colors
OVERLAY_PROTEIN_COLOR = [255, 0, 0]    # RGB for protein signal layer
OVERLAY_VESSEL_COLOR = [255, 255, 0]   # RGB for vessel outline layer
OVERLAY_DEXTRAN_COLOR = [255, 0, 255]  # RGB for dextran signal layer

AUTO_MODE = False
AUTO_THRESHOLD_METHOD = 'Otsu'
BG_CORRECTION = True
BG_FRACTION = 0.15

# channel auto-detection from CZI metadata
AUTO_DETECT_CHANNELS = True

# fallback config (only if AUTO_DETECT_CHANNELS=False or detection fails)
CHANNEL_CONFIG = {
    'OLD':   {'protein': 1, 'TL': 2, 'dextran': 3, 'DAPI': 4},
    'AD':    {'protein': 1, 'TL': 2, 'dextran': 3, 'DAPI': 4},
    'YOUNG': {'protein': 1, 'TL': 2, 'dextran': 0, 'DAPI': 3},
    'TBI':   {'protein': 1, 'TL': 2, 'dextran': 0, 'DAPI': 3}
}

MAGNIFICATIONS = ['20x', '40x', '63x']
SCALE_BAR_SIZES = {
    '20x': 100,
    '40x': 50,
    '63x': 20,
}

In [ ]:
# HELPERS

# extract single channel from CZI as z-stack array (1-indexed channel number)
def extract_channel_zstack(czi_path, channel_number):
    if channel_number == 0:
        return None
    ch_idx = channel_number - 1
    img = BioImage(str(czi_path))
    data = img.data
    dims = img.dims.order
    if 'C' in dims:
        data = np.take(data, ch_idx, axis=dims.index('C'))
    return np.squeeze(data)


# max-intensity projection of a single channel
def extract_channel_mip(czi_path, channel_number):
    zstack = extract_channel_zstack(czi_path, channel_number)
    if zstack is None:
        return None
    return np.max(zstack, axis=0) if zstack.ndim == 3 else zstack


# pad or trim z-stack to exact target_z slices
def pad_zstack(data, target_z):
    if data.ndim == 2:
        padded = np.zeros((target_z, data.shape[0], data.shape[1]), dtype=data.dtype)
        padded[0] = data
        return padded
    current_z = data.shape[0]
    if current_z == target_z:
        return data
    elif current_z > target_z:
        return data[:target_z]
    else:
        pad_width = [(0, target_z - current_z)] + [(0, 0)] * (data.ndim - 1)
        return np.pad(data, pad_width, mode='constant', constant_values=0)


# save z-stack as gzip-compressed HDF5
def save_zstack_h5(data, path):
    with h5py.File(str(path), 'w') as f:
        f.create_dataset('data', data=data.astype(np.uint16), compression='gzip')


# extract mouse type from filename (matches MOUSE_TYPES list)
def detect_mouse_type(filename):
    fname = filename.upper()
    for mt in MOUSE_TYPES:
        if f'_{mt}_' in fname or fname.startswith(f'{mt}_'):
            return mt
    return 'UNKNOWN'


# extract brain region from filename (matches BRAIN_REGIONS list)
def detect_region(filename):
    fname = filename.upper()
    for r in BRAIN_REGIONS:
        if r in fname:
            return r
    return 'UNKNOWN'


# check if file matches TARGET_MOUSE_TYPES and TARGET_REGIONS filters
def is_target(filename):
    if TARGET_MOUSE_TYPES is None and TARGET_REGIONS is None:
        return True
    mt = detect_mouse_type(filename)
    reg = detect_region(filename)
    mt_ok = (TARGET_MOUSE_TYPES is None) or (mt in TARGET_MOUSE_TYPES)
    reg_ok = (TARGET_REGIONS is None) or (reg in TARGET_REGIONS)
    return mt_ok and reg_ok


# assign channels by excitation wavelength: 488=TL, 514-561=protein, 590-640=dextran, 405=DAPI
def get_channel_config(czi_path):
    img = BioImage(str(czi_path))
    ome = img.ome_metadata
    channels = ome.images[0].pixels.channels
    config = {'protein': 0, 'TL': 0, 'dextran': 0, 'DAPI': 0}
    for i, ch in enumerate(channels):
        ex = float(ch.excitation_wavelength) if ch.excitation_wavelength else 0
        fl = (ch.fluor or '').lower()
        ch_num = i + 1
        if 480 <= ex <= 495 or 'alexa fluor 488' in fl:
            config['TL'] = ch_num
        elif 510 <= ex <= 565 or 'alexa fluor 555' in fl:
            config['protein'] = ch_num
        elif 590 <= ex <= 640 or 'texas red' in fl:
            config['dextran'] = ch_num
        elif 400 <= ex <= 410 or 'dapi' in fl:
            config['DAPI'] = ch_num
    return config


# cached wrapper for get_channel_config (avoids re-reading same CZI)
_channel_cache = {}
def get_channel_config_cached(czi_path):
    if czi_path not in _channel_cache:
        _channel_cache[czi_path] = get_channel_config(czi_path)
    return _channel_cache[czi_path]


# normalize array to 0-1 range
def norm(img):
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else np.zeros_like(img)


# convert grayscale 0-1 to green RGB uint8
def make_green(gray):
    rgb = np.zeros((*gray.shape, 3), dtype=np.uint8)
    rgb[..., 1] = (gray * 255).astype(np.uint8)
    return rgb


# auto-detect channel roles from CZI metadata, print assignments
def detect_channels_from_metadata(image_path):
    config = get_channel_config(str(image_path))
    if config['protein'] > 0 and config['TL'] > 0:
        detected = []
        for role, ch in config.items():
            if ch > 0:
                detected.append(f'C{ch}={role}')
        print(f'    Auto-detected: {" | ".join(detected)}')
        return config
    else:
        print(f'    [!] Could not identify protein+TL from metadata')
        return None


# read objective magnification from CZI XML metadata
def get_magnification_from_metadata(czi_path):
    img = BioImage(str(czi_path))
    meta = img.metadata
    for elem in meta.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        text = (elem.text or '').strip()
        if tag == 'NominalMagnification' and text:
            mag = text.replace('.0', '')
            return f"{mag}x"
    # fallback: check OME objective
    ome = img.ome_metadata
    if ome.instruments:
        for obj in ome.instruments[0].objectives:
            if obj.nominal_magnification:
                return f"{int(obj.nominal_magnification)}x"
    return None


In [ ]:
# ROI Processor

#process each image with standardized naming convention
class IndividualROIProcessor:
    
    def __init__(self, fiji_path, output_dir, debug_mode=False, use_roi=True, 
                 show_background_ui=False, auto_save_images=False,
                 auto_mode=False, auto_threshold_method='Otsu',
                 bg_correction=True, bg_fraction=0.3,
                 reuse_rois=False,
                 ilastik_tl_mode='off'):
        self.fiji_path = fiji_path
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.debug_mode = debug_mode
        self.use_roi = use_roi
        self.show_background_ui = show_background_ui
        self.auto_save_images = auto_save_images
        self.auto_mode = auto_mode
        self.auto_threshold_method = auto_threshold_method
        self.bg_correction = bg_correction
        self.bg_fraction = bg_fraction
        self.reuse_rois = reuse_rois
        self.blinded = True
        self.blind_map = {}
        self.ilastik_tl_mode = ilastik_tl_mode
        
        # Auto mode overrides: force settings that conflict with full automation
        if self.auto_mode:
            self.use_roi = False          # can't draw ROIs in auto mode
            self.show_background_ui = False
            self.debug_mode = False
        
        # Reuse ROIs: vessel mask reuse doesn't need freehand ROI pipeline
        # use_roi stays as configured (False = whole image)
        
        # Create organized directory structure
        self._create_directory_structure()
        
        self.temp_dir = self.output_dir / "temp"
        self.temp_dir.mkdir(exist_ok=True)
        
        # Create directory for saved images
        self.images_dir = self.output_dir / "processed_images"
        self.images_dir.mkdir(exist_ok=True)
        
        # Results storage
        self.results_csv = self.output_dir / "vessel_analysis_results.csv"
        self.all_results = []
        
        # Find Fiji executable
        self.fiji_exe = self._find_fiji_exe()
        if not self.fiji_exe:
            raise ValueError(f"Could not find Fiji executable in {fiji_path}")
        
        print(f"[OK] Found Fiji: {self.fiji_exe}")
        if self.auto_mode:
            print(f"[FULL AUTO MODE] - No user prompts after channel identification")
            print(f"                 - Threshold method: {self.auto_threshold_method}")
            print(f"                 - ROI disabled (whole image), debug disabled")
        if self.debug_mode:
            print("[DEBUG MODE ENABLED] - Will pause at each processing step")
            print("                     - Only processing only 1 image per mouse type")
        if not self.use_roi:
            print("[WHOLE IMAGE MODE] - ROI selection disabled, analyzing full images")
        if self.show_background_ui:
            print("[MANUAL BACKGROUND] - Background subtraction UI will be shown")
        if self.auto_save_images:
            print("[AUTO SAVE] - All processed images will be saved automatically")
        if self.reuse_rois:
            print("[REUSE ROIs] - Loading previously saved ROIs instead of manual drawing")
        if self.bg_correction:
            print(f"[BG CORRECTION] - Adaptive threshold: bg_median + {self.bg_fraction*100:.0f}% of dynamic range")
        else:
            print("[NO BG CORRECTION] - Using Otsu/manual threshold on overlay")
    
    def _should_use_ilastik(self, mouse_type, image_path):
        if self.reuse_rois:
            return False  # vessel masks already decided
        if self.ilastik_tl_mode == 'off':
            return False
        stem = os.path.splitext(os.path.basename(str(image_path)))[0]
        mask_dir = Path(str(image_path)).parent / "vessel_masks"
        weighted_path = mask_dir / f"{stem}_TL_dominant.tif"
        if not weighted_path.exists():
            return False
        mt = mouse_type.upper()
        if self.ilastik_tl_mode == 'auto':
            return mt in ['AD', 'TBI', 'OLD']
        elif self.ilastik_tl_mode == 'manual':
            return 'prompt'
        return False

    # create organized directory structure based on naming convention
    def _create_directory_structure(self): 
        
        # Main directories
        self.roi_dir = self.output_dir / "ROIs"
        self.roi_dir.mkdir(exist_ok=True)
        
        # By Mouse Type directories
        self.by_mouse_dir = self.output_dir / "By_Mouse_Type"
        self.by_mouse_dir.mkdir(exist_ok=True)
        
        # By Brain Region directories
        self.by_region_dir = self.output_dir / "By_Brain_Region"
        self.by_region_dir.mkdir(exist_ok=True)
        
        # By Protein directories
        self.by_protein_dir = self.output_dir / "By_Protein"
        self.by_protein_dir.mkdir(exist_ok=True)
        
        # Create subdirectories for each category
        for mouse_type in MOUSE_TYPES:
            mouse_dir = self.by_mouse_dir / mouse_type
            mouse_dir.mkdir(exist_ok=True)
            
            # Create region subdirs within each mouse type
            for region in BRAIN_REGIONS:
                region_subdir = mouse_dir / region
                region_subdir.mkdir(exist_ok=True)
        
        for region in BRAIN_REGIONS:
            region_dir = self.by_region_dir / region
            region_dir.mkdir(exist_ok=True)
        
        for protein in PROTEINS:
            protein_dir = self.by_protein_dir / protein
            protein_dir.mkdir(exist_ok=True)
    
    #search given path for Fiji.app folder
    def _find_fiji_exe(self): 
        possible = ["ImageJ-win64.exe", "ImageJ-win32.exe", "fiji-win64.exe", "ImageJ.exe"]
        for exe in possible:
            full_path = os.path.join(self.fiji_path, exe)
            if os.path.exists(full_path):
                return full_path
        return None

    #parse the filename format (should be standardized?)
    def _parse_standardized_filename(self, filename): 

        # Remove file extension if present
        filename_base = filename.replace('.czi', '').replace('.tif', '')
        
        # Split by underscore
        parts = filename_base.split('_')
        
        # Initialize with defaults
        parsed = {
            'protein': 'unknown',
            'magnification': 'unknown',
            'mouse_type': 'unknown',
            'brain_region': 'unknown',
            'sample_number': 'unknown',
            'mouse_id': 'unknown',
            'has_dextran': False
        }
        
        # parse based on expected format
        if len(parts) >= 6:
            parsed['protein'] = parts[0].upper()
            # parts[1] should be 'TL'
            parsed['magnification'] = parts[2]
            parsed['mouse_type'] = parts[3].upper()
            parsed['brain_region'] = parts[4].upper()
            parsed['sample_number'] = parts[5]
            
            # Create mouse ID (based on type and number [ex. AD_1])
            parsed['mouse_id'] = f"{parsed['mouse_type']}_{parsed['sample_number']}"
            
            # check if it's old or AD to figure out if it has dextran
            mouse_type_lower = parsed['mouse_type'].lower()
            parsed['has_dextran'] = mouse_type_lower in ['old', 'ad']
        elif len(parts) >= 5:
            parsed['protein'] = parts[0].upper()
            # parts[1] should be 'TL'
            parsed['magnification'] = parts[1]
            parsed['mouse_type'] = parts[2].upper()
            parsed['brain_region'] = parts[3].upper()
            parsed['sample_number'] = parts[4]
            
            # Create mouse ID (based on type and number [ex. AD_1])
            parsed['mouse_id'] = f"{parsed['mouse_type']}_{parsed['sample_number']}"
            
            # check if it's old or AD to figure out if it has dextran
            mouse_type_lower = parsed['mouse_type'].lower()
            parsed['has_dextran'] = mouse_type_lower in ['old', 'ad']        
        else:
            # fallback parsing for non-standard names
            filename_lower = filename_base.lower()
            
            # try to find mouse type
            for mouse_type in MOUSE_TYPES:
                if mouse_type.lower() in filename_lower:
                    parsed['mouse_type'] = mouse_type.upper()
                    break
            
            # Try to find brain region
            for region in BRAIN_REGIONS:
                if region.lower() in filename_lower:
                    parsed['brain_region'] = region.upper()
                    break
            
            # try to find protein
            for protein in PROTEINS:
                if protein.lower() in filename_lower:
                    parsed['protein'] = protein.upper()
                    break
            
            # try to find magnification
            mag_match = re.search(r'(\d+)x', filename_lower)
            if mag_match:
                parsed['magnification'] = mag_match.group(0)
            
            # try to find sample number
            num_match = re.search(r'_(\d+)$', filename_base)
            if num_match:
                parsed['sample_number'] = num_match.group(1)
                parsed['mouse_id'] = f"{parsed['mouse_type']}_{parsed['sample_number']}"
            
            # check for dextran
            parsed['has_dextran'] = parsed['mouse_type'].lower() in ['old', 'ad']

        # If protein is still unknown, use first part of filename
        if parsed['protein'] == 'unknown' and len(parts) > 0:
            parsed['protein'] = parts[0].upper()

        
        return parsed

    # sort images by mouse type, brain region, and sample number
    def _sort_images(self, image_files): 
        
        # create a list with metadata
        images_with_metadata = []
        for img_path in image_files:
            filename = img_path.stem
            metadata = self._parse_standardized_filename(filename)
            images_with_metadata.append({
                'path': img_path,
                'filename': filename,
                'protein': metadata['protein'],
                'magnification': metadata['magnification'],
                'mouse_type': metadata['mouse_type'],
                'brain_region': metadata['brain_region'],
                'sample_number': metadata['sample_number'],
                'mouse_id': metadata['mouse_id'],
                'has_dextran': metadata['has_dextran']
            })
        
        # sort by: mouse_type > brain_region > sample_number > protein
        def sort_key(x):
            
            # define sort order using the MOUSE_TYPES order
            mouse_order = {mt.upper(): i for i, mt in enumerate(MOUSE_TYPES)}
            region_order = {r.upper(): i for i, r in enumerate(BRAIN_REGIONS)}
            
            # convert sample number to int for proper sorting
            try:
                sample_num = int(x['sample_number'])
            except ValueError:
                sample_num = 999
            
            return (
                mouse_order.get(x['mouse_type'], 999),
                region_order.get(x['brain_region'], 999),
                sample_num,
                x['protein']
            )
        
        images_with_metadata.sort(key=sort_key)
        
        return images_with_metadata

    # process a single image with updated workflow
    def process_single_image_with_roi(self, image_data, image_index, total_images):
        
        image_path = image_data['path']
        filename = image_data['filename']
        protein = image_data['protein']
        magnification = image_data['magnification']
        mouse_type = image_data['mouse_type']
        brain_region = image_data['brain_region']
        sample_number = image_data['sample_number']
        mouse_id = image_data['mouse_id']
        has_dextran = image_data['has_dextran']
        blind_id = self.blind_map.get(filename, filename)
        
        if self.blinded:
            print(f"\n[{image_index}/{total_images}] Processing: {blind_id}")
        else:
            print(f"\n[{image_index}/{total_images}] Processing: {filename}")
            print(f"  Protein: {protein} | Mouse: {mouse_id} | Region: {brain_region}")
        # use metadata magnification if available, fall back to filename
        meta_mag = get_magnification_from_metadata(image_path)
        if meta_mag and meta_mag in SCALE_BAR_SIZES:
            magnification = meta_mag
        
        # Detect channels from metadata
        channel_config = None
        if AUTO_DETECT_CHANNELS:
            channel_config = detect_channels_from_metadata(image_path)
        
        if channel_config is None:
            channel_config = CHANNEL_CONFIG.get(mouse_type.upper(), CHANNEL_CONFIG.get('YOUNG', {'protein': 1, 'TL': 2, 'dextran': 0, 'DAPI': 0}))
            print(f"    Using fallback config: {channel_config}")
        
        # Update has_dextran based on detected channels
        has_dextran = channel_config.get('dextran', 0) > 0
        
        print("  -> Image opening in Fiji...")
        if self.auto_mode:
            print(f"  -> [AUTO MODE] Fully automatic processing ({self.auto_threshold_method} threshold)")
        elif self.reuse_rois:
            print("  -> [REUSE ROIs] Loading saved ROI, re-running analysis")
        elif not self.use_roi:
            print("  -> Whole Image Analysis - No ROI selection; Analyzing full image")
        elif not self.debug_mode:
            print("  -> Draw FREEHAND ROI and click OK")
        else:
            print("  -> [DEBUG MODE] Step-by-step processing enabled")
            print("  -> Click Cancel at any debug dialog to abort current image")
        

        # resolve ilastik TL path
        use_ilastik = self._should_use_ilastik(mouse_type, image_path)
        ilastik_tl_path = None
        if use_ilastik is not False:
            from pathlib import Path as _P
            _stem = os.path.splitext(os.path.basename(str(image_path)))[0]
            _wpath = _P(str(image_path)).parent / "vessel_masks" / f"{_stem}_TL_dominant.tif"
            if _wpath.exists():
                ilastik_tl_path = str(_wpath).replace("\\", "/")
                if use_ilastik == True:
                    print(f"  -> ilastik TL: using weighted TL ({self.ilastik_tl_mode} mode)")
                elif use_ilastik == 'prompt':
                    print(f"  -> ilastik TL: will prompt ({self.ilastik_tl_mode} mode)")

        # look up saved dextran threshold from previous results
        saved_dex_threshold = 0
        if self.reuse_rois and self.results_csv.exists():
            prev_df = pd.read_csv(self.results_csv)
            match = prev_df[prev_df['filename'] == filename]
            if not match.empty and 'dextran_threshold' in match.columns:
                saved_dex_threshold = float(match.iloc[0]['dextran_threshold'])
                if saved_dex_threshold > 0:
                    print(f"  -> Saved dextran threshold: {saved_dex_threshold:.0f}")

        # Create macro for this specific image
        macro = self._create_single_image_macro(
            str(image_path), 
            filename, 
            has_dextran,
            protein,
            magnification,
            mouse_type,
            brain_region,
            sample_number,
            mouse_id,
            image_index,
            total_images,
            channel_config,
            ilastik_tl_path,
            ilastik_tl_prompt=(use_ilastik == 'prompt'),
            saved_dex_threshold=saved_dex_threshold,
            blind_id=blind_id
        )
        
        # save macro
        macro_label = blind_id if self.blinded else filename
        macro_file = self.temp_dir / f"process_{macro_label}.ijm"
        with open(macro_file, 'w', encoding='utf-8') as f:
            f.write(macro)
        
        # open FIJI
        cmd = [self.fiji_exe, "-macro", str(macro_file)]
        
        try:
            # run and wait for completion
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=1200)  # will cancel after 2 minutes; increased for manual steps
            
            # check if macro was cancelled by buttons 
            if "User canceled the macro" in result.stderr or "Macro canceled" in result.stdout:
                print("Cancelled processing for this image")
                return self._create_error_result(filename, protein, magnification, 
                                                mouse_type, brain_region, sample_number, 
                                                mouse_id, 'User cancelled')
            
            # read results for this image
            temp_result_file = self.temp_dir / f"{blind_id if self.blinded else filename}_results.txt"
            
            if temp_result_file.exists():
                # read results
                with open(temp_result_file, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                
                # Parse results
                results = {
                    'filename': filename,
                    'protein': protein,
                    'magnification': magnification,
                    'mouse_type': mouse_type,
                    'brain_region': brain_region,
                    'sample_number': sample_number,
                    'mouse_id': mouse_id
                }
                
                for line in lines:
                    if ':' in line:
                        key, value = line.strip().split(':', 1)
                        try:
                            results[key] = float(value)
                        except ValueError:
                            results[key] = value
                
                # clean temp file
                temp_result_file.unlink()
                
                # display results
                print(f"  [OK] Analysis complete:")
                print(f"    - Magnification: {results.get('meta_magnification', 'unknown')} | Pixel: {results.get('meta_pixel_size_um', 0)} um/px")
                print(f"    - Vessel area: {results.get('vessel_area_percent', 0):.1f}%")
                print(f"    - MFI: {results.get('mean_intensity', 0):.1f}")
                if results.get('bg_threshold', 0) > 0:
                    print(f"    - BG threshold: {results.get('bg_threshold', 0):.2f} (median={results.get('bg_median', 0):.2f} + {self.bg_fraction*100:.0f}% range)")
                print(f"    - Protein coverage on vessels: {results.get('percent_coverage_on_vessels', 0):.1f}%")
                if has_dextran:
                    print(f"    - Dextran (parenchymal): {results.get('dextran_coverage_percent', 0):.1f}%")
                    print(f"    - Dextran (total): {results.get('dextran_total_percent', 0):.1f}%")
                
                return results
            else:
                print("  [!] No results generated - image may have been skipped or cancelled")
                return self._create_error_result(filename, protein, magnification, 
                                                mouse_type, brain_region, sample_number, 
                                                mouse_id, 'No results generated')
                
        except subprocess.TimeoutExpired:
            print("  [!] Processing timeout - moving to next image")
            return self._create_error_result(filename, protein, magnification, 
                                            mouse_type, brain_region, sample_number, 
                                            mouse_id, 'Timeout')
        except Exception as e:
            print(f"  [X] Error: {e}")
            return self._create_error_result(filename, protein, magnification, 
                                            mouse_type, brain_region, sample_number, 
                                            mouse_id, str(e))
    
    # create an error result dict
    def _create_error_result(self, filename, protein, magnification, mouse_type, 
                           brain_region, sample_number, mouse_id, error):
        return {
            'filename': filename,
            'protein': protein,
            'magnification': magnification,
            'mouse_type': mouse_type,
            'brain_region': brain_region,
            'sample_number': sample_number,
            'mouse_id': mouse_id,
            'error': error
        }

    # create macro for processing a single image with updated workflow    
    def _create_single_image_macro(self, image_path, filename, has_dextran, 
                                   protein, magnification, mouse_type, 
                                   brain_region, sample_number, mouse_id,
                                   index, total, channel_config,
                                   ilastik_tl_path=None, ilastik_tl_prompt=False,
                                   saved_dex_threshold=0, blind_id=''):
        
        # i need to fix paths for ImageJ
        image_path = image_path.replace("\\", "/")
        output_dir = str(self.output_dir).replace("\\", "/")
        temp_dir = str(self.temp_dir).replace("\\", "/")
        roi_dir = str(self.roi_dir).replace("\\", "/")
        images_dir = str(self.images_dir).replace("\\", "/")
        
        # channel_config is now passed in from auto-detection or fallback
        
        # get scale bar size for this magnification from config
        scale_bar_size = SCALE_BAR_SIZES.get(magnification, 50)

        # macro section
        # when blinded, close the Log window at start of macro
        log_suppress = 'print("\\\\Clear"); if (isOpen("Log")) {{ selectWindow("Log"); run("Close"); }}\n' if self.blinded else ''
        macro = log_suppress + f"""
// UPDATED WORKFLOW - Process image {'with freehand ROI' if self.use_roi else '- WHOLE IMAGE'}
// Image: {"" if self.blinded else filename}
// {"BLINDED" if self.blinded else "Mouse: " + mouse_id + " | Region: " + brain_region + " | Protein: " + protein}
// DEBUG MODE: {"ENABLED" if self.debug_mode else "DISABLED"}
// ROI MODE: {"ENABLED" if self.use_roi else "DISABLED - Whole Image"}
// BACKGROUND UI: {"MANUAL" if self.show_background_ui else "AUTOMATIC"}
// AUTO MODE: {"ENABLED - " + self.auto_threshold_method + " threshold" if self.auto_mode else "DISABLED - Manual thresholds"}
// BG CORRECTION: {"ENABLED - mean + " + str(int(self.bg_fraction*100)) + "% range" if self.bg_correction else "DISABLED"}
// CHANNEL DETECTION: auto-detected from metadata


// Initialize variables for results
vessel_area = 0;
overlay_mfi = 0;
protein_coverage = 0;
percent_on_vessels = 0;
dextran_coverage = 0;
dextran_total = 0;
dextran_threshold = 0;
bg_mean = 0;
bg_median = 0;
bg_std = 0;
bg_threshold = 0;
protein_max = 0;
roi_x = 0;
roi_y = 0;
roi_width = 0;
roi_height = 0;
roi_area_actual = 0;
original_width = 0;
original_height = 0;
meta_magnification = "unknown";
meta_pixel_size_um = 0;
meta_tile_scan = "No";
meta_objective = "unknown";
meta_bit_depth = 0;

// STEP 1: Import image with Bio-Formats
print("Opening: {filename}");"""

        if self.debug_mode:
            macro += f"""
// Debug dialog with Cancel option
Dialog.create("DEBUG - Opening Image");
Dialog.addMessage("About to open image:");
Dialog.addMessage("{filename}");

Dialog.addMessage("Click OK to open the image");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();  // If Cancel is clicked, macro stops here"""

        macro += f"""
run("Bio-Formats", "open=[{image_path}] color_mode=Composite rois_import=[ROI manager] view=Hyperstack stack_order=XYCZT");
title = getTitle();"""

        if self.debug_mode:
            macro += f"""
// Debug dialog with Cancel option
Dialog.create("DEBUG - Image Opened");
Dialog.addMessage("Image has been opened.");
Dialog.addMessage("Check the channels are displayed correctly.");

Dialog.addMessage("Click OK to continue");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();  // If Cancel is clicked, macro stops here"""

        macro += f"""

// Get dimensions
getDimensions(width, height, channels, slices, frames);
original_width = width;
original_height = height;

print("  Image dimensions: " + width + "x" + height);
print("  Channels: " + channels + ", Slices: " + slices + ", Frames: " + frames);"""

        # METADATA EXTRACTION - uses regular string (not f-string) to avoid brace escaping issues
        macro += """

// EXTRACT METADATA (magnification, pixel size, tile scan)
meta_bit_depth = bitDepth();

// Get pixel size from calibration
getPixelSize(unit, pw, ph);
if (unit == "micron" || unit == "um" || unit == "\u00B5m" || unit == "microns") {
    meta_pixel_size_um = pw;
} else {
    meta_pixel_size_um = 0;
}

// Parse image info for magnification and objective
info_str = getMetadata("Info");
if (info_str == "") {
    info_str = getImageInfo();
}

// Search for magnification
meta_magnification = "unknown";
meta_objective = "unknown";

mag_keys = newArray("Nominal Magnification", "NominalMagnification", "Magnification",
                    "Objective Magnification", "objective.nominal_magnification");
for (mk = 0; mk < mag_keys.length; mk++) {
    idx = indexOf(info_str, mag_keys[mk]);
    if (idx >= 0) {
        sub = substring(info_str, idx);
        eq_pos = indexOf(sub, "=");
        if (eq_pos < 0) eq_pos = indexOf(sub, ":");
        if (eq_pos >= 0) {
            num_start = eq_pos + 1;
            while (num_start < lengthOf(sub) && substring(sub, num_start, num_start+1) == " ") {
                num_start++;
            }
            num_end = num_start;
            found_end = false;
            while (num_end < lengthOf(sub) && !found_end) {
                c = substring(sub, num_end, num_end+1);
                if (c == "." || (c >= "0" && c <= "9")) {
                    num_end++;
                } else {
                    found_end = true;
                }
            }
            if (num_end > num_start) {
                mag_val = substring(sub, num_start, num_end);
                meta_magnification = mag_val + "x";
            }
        }
        mk = mag_keys.length; // break
    }
}

// Search for objective name
obj_keys = newArray("Objective Name", "ObjectiveName", "objective.name", "Lens Name");
for (ok = 0; ok < obj_keys.length; ok++) {
    idx = indexOf(info_str, obj_keys[ok]);
    if (idx >= 0) {
        sub = substring(info_str, idx);
        eq_pos = indexOf(sub, "=");
        if (eq_pos < 0) eq_pos = indexOf(sub, ":");
        if (eq_pos >= 0) {
            val_start = eq_pos + 1;
            while (val_start < lengthOf(sub) && substring(sub, val_start, val_start+1) == " ") {
                val_start++;
            }
            val_end = val_start;
            found_end = false;
            while (val_end < lengthOf(sub) && !found_end) {
                c = substring(sub, val_end, val_end+1);
                if (c == "|") {
                    found_end = true;
                } else {
                    val_end++;
                }
            }
            if (val_end > val_start) {
                meta_objective = substring(sub, val_start, minOf(val_end, val_start + 80));
            }
        }
        ok = obj_keys.length; // break
    }
}

// Detect tile scan
meta_tile_scan = "No";
if (indexOf(info_str, "TileScan") >= 0 || indexOf(info_str, "Tile Scan") >= 0) {
    meta_tile_scan = "TileScan";
} else if (indexOf(info_str, "Scene") >= 0) {
    scene_idx = indexOf(info_str, "Scene #");
    if (scene_idx >= 0) {
        meta_tile_scan = "MultiScene";
    }
}

print("  Metadata:");
print("    Magnification: " + meta_magnification);
print("    Objective: " + meta_objective);
print("    Pixel size: " + d2s(meta_pixel_size_um, 4) + " um/px");
print("    Bit depth: " + meta_bit_depth);
print("    Tile scan: " + meta_tile_scan);
"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Image Info");
Dialog.addMessage("Image dimensions: " + width + "x" + height);
Dialog.addMessage("Channels: " + channels);
Dialog.addMessage("Slices: " + slices);
Dialog.addMessage("Frames: " + frames);

Dialog.addMessage("Click OK to continue or Cancel to skip");
Dialog.show();"""

        macro += f"""

// STEP 2: CHANNEL ASSIGNMENT (auto-detected from metadata)

protein_channel = {channel_config["protein"]};
tl_channel = {channel_config["TL"]};
dextran_channel = {channel_config.get("dextran", 0)};
dapi_channel = {channel_config.get("DAPI", 0)};

print("  Channels (auto-detected):");
print("    Protein ({protein}): C" + protein_channel);
print("    Tomato Lectin: C" + tl_channel);
if (dextran_channel > 0) print("    Dextran: C" + dextran_channel);
if (dapi_channel > 0) print("    DAPI: C" + dapi_channel);

// Build channel prefixes for later use
protein_channel_prefix = "C" + protein_channel + "-";
tl_channel_prefix = "C" + tl_channel + "-";
if (dextran_channel > 0) {{
    dextran_channel_prefix = "C" + dextran_channel + "-";
}} else {{
    dextran_channel_prefix = "";
}}
if (dapi_channel > 0) {{
    dapi_channel_prefix = "C" + dapi_channel + "-";
}} else {{
    dapi_channel_prefix = "";
}}
"""

        if self.auto_mode:
            macro += f"""
// [AUTO MODE] Enable batch mode
setBatchMode(true);
print("  [AUTO] Batch mode enabled");
"""

        macro += f"""

// STEP 4: Z-stack with max intensity projection (AFTER channel verification)
if (slices > 1) {{
    print("  Creating max projection...");"""
    
        if self.debug_mode:
            macro += f"""
    Dialog.create("DEBUG - Z-Projection");
    Dialog.addMessage("About to create MAX intensity Z-projection");
    Dialog.addMessage("Image has " + slices + " slices");
    
    Dialog.addMessage("Click OK to create projection");
    Dialog.addMessage("Click Cancel to skip this image");
    Dialog.show();"""
    
        macro += f"""
    run("Z Project...", "projection=[Max Intensity]");
    close(title);
    title = getTitle();"""
    
        if self.debug_mode:
            macro += f"""
    Dialog.create("DEBUG - Z-Projection Complete");
    Dialog.addMessage("Z-projection complete.");
    Dialog.addMessage("Original stack closed.");
    
    Dialog.addMessage("Click OK to continue or Cancel to skip");
    Dialog.show();"""
    
        macro += f"""
}}

// STEP 5: Background subtraction"""

        if self.show_background_ui:
            macro += f"""
print("  Manual background subtraction...");
run("Subtract Background...");
waitForUser("Background Subtraction", 
    "The Subtract Background dialog is now open.\\n\\n" +
    "1. Adjust rolling ball radius (default: {ROLLING_BALL_RADIUS})\\n" +
    "2. Check 'Sliding paraboloid' if desired\\n" +
    "3. Check 'Stack' to apply to all channels\\n" +
    "4. Click OK in the dialog\\n" +
    "5. Then click OK in this message to continue");"""
        else:
            macro += f"""
print("  Subtracting background automatically...");"""
            
            if self.debug_mode:
                macro += f"""
Dialog.create("DEBUG - Background Subtraction");
Dialog.addMessage("About to subtract background");
Dialog.addMessage("Rolling ball radius: {ROLLING_BALL_RADIUS}");
Dialog.addMessage("Sliding paraboloid enabled");

Dialog.addMessage("Click OK to subtract background");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""
            
            macro += f"""
run("Subtract Background...", "rolling={ROLLING_BALL_RADIUS} sliding stack");"""
            
        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Background Subtracted");
Dialog.addMessage("Background subtraction complete.");
Dialog.addMessage("Check the result.");

Dialog.addMessage("Click OK to continue");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""


// {'FREEHAND ROI SELECTION' if self.use_roi else 'WHOLE IMAGE ANALYSIS'} FOR THIS IMAGE
"""
        
        if self.use_roi:

            if self.reuse_rois:
                # Load saved ROI instead of manual drawing
                macro += f"""
// REUSE SAVED ROI
// try zip first (multi-ROI), then single .roi
roi_path = "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}_rois.zip";
if (!File.exists(roi_path)) {{
    roi_path = "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}.roi";
}}
if (File.exists(roi_path)) {{
    print("  Loading saved ROI: " + roi_path);
    roiManager("Reset");
    roiManager("Open", roi_path);
    nROIs = roiManager("count");
    if (nROIs > 1) {{
        indices = newArray(nROIs);
        for (ri = 0; ri < nROIs; ri++) indices[ri] = ri;
        roiManager("Select", indices);
        roiManager("Combine");
    }} else {{
        roiManager("Select", 0);
    }}
    
    getSelectionBounds(roi_x, roi_y, roi_width, roi_height);
    getStatistics(roi_area_actual);
    print("  " + nROIs + " ROI(s) loaded, area: " + roi_area_actual + " pixels");
}} else {{
    print("  No saved ROI found - using full image");
    roi_x = 0;
    roi_y = 0;
    roi_width = width;
    roi_height = height;
    roi_area_actual = width * height;
}}
"""

            else:
                # Multi-ROI drawing - split channels so user can see TL
                macro += f"""
// SPLIT CHANNELS FOR ROI DRAWING
// duplicate composite, split to show TL clearly
run("Select None");
run("Duplicate...", "duplicate");
rename("roi_display");
run("Split Channels");

// find and enhance the TL channel for drawing
list = getList("image.titles");
tl_display = "";
for (i = 0; i < list.length; i++) {{
    if (indexOf(list[i], "C" + tl_channel + "-") >= 0) {{
        tl_display = list[i];
    }}
}}

if (tl_display != "") {{
    selectWindow(tl_display);
    run("Green");
    run("Enhance Contrast", "saturated=0.35");
    run("Maximize");
    rename("TL (draw ROIs here)");
}}

// ROI SELECTION
roiManager("Reset");
setTool("freehand");
waitForUser("ROI Selection - Image {index}/{total}", 
    "{"Image: " + blind_id if self.blinded else f"File: {filename}"}\\n" +
    "{"" if self.blinded else f"Mouse: {mouse_id} | Region: {brain_region} | Protein: {protein}"}\\n" +
    "\\n" +
    "Draw ROIs on the TL channel (green):\\n" +
    "1. Draw freehand around a vessel or group\\n" +
    "2. Press T to add it to ROI Manager\\n" +
    "3. Repeat for all vessels you want to include\\n" +
    "4. Click OK when done\\n\\n" +
    "If no ROIs are added, the full image will be analyzed.");

// check what we have
nROIs = roiManager("count");

if (nROIs == 0 && selectionType() != -1) {{
    // user drew one ROI but didn't press T
    roiManager("Add");
    nROIs = 1;
}}

// close ROI display windows only
list = getList("image.titles");
for (i = list.length - 1; i >= 0; i--) {{
    win = list[i];
    if (indexOf(win, "draw ROIs") >= 0 || indexOf(win, "roi_display") >= 0) {{
        selectWindow(win);
        close();
    }}
}}
// close split channel windows from display
list = getList("image.titles");
for (i = list.length - 1; i >= 0; i--) {{
    win = list[i];
    if (startsWith(win, "C") && indexOf(win, "-") > 0 && win != title) {{
        selectWindow(win);
        close();
    }}
}}
selectWindow(title);

if (nROIs == 0) {{
    // no ROIs at all - use full image
    print("  No ROIs drawn - analyzing full image");
    makeRectangle(0, 0, width, height);
    roi_area_actual = width * height;
    roi_x = 0;
    roi_y = 0;
    roi_width = width;
    roi_height = height;
}} else if (nROIs == 1) {{
    // single ROI
    roiManager("Select", 0);
    getSelectionBounds(roi_x, roi_y, roi_width, roi_height);
    getStatistics(roi_area_actual);
    print("  1 ROI selected, area: " + roi_area_actual + " pixels");
}} else {{
    // multiple ROIs - combine into one selection
    indices = newArray(nROIs);
    for (ri = 0; ri < nROIs; ri++) indices[ri] = ri;
    roiManager("Select", indices);
    roiManager("Combine");
    getSelectionBounds(roi_x, roi_y, roi_width, roi_height);
    getStatistics(roi_area_actual);
    print("  " + nROIs + " ROIs combined, total area: " + roi_area_actual + " pixels");
}}

// save ROIs (if any were drawn)
if (roiManager("count") > 0) {{
    roiManager("Save", "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}_rois.zip");
    print("  ROIs saved");
}}
"""

            if self.debug_mode:
                macro += f"""
Dialog.create("DEBUG - ROI Saved");
Dialog.addMessage("Freehand ROI has been saved.");
Dialog.addMessage("ROI area: " + roi_area_actual + " pixels");

Dialog.addMessage("About to:");
Dialog.addMessage("1. Duplicate image with ROI");
Dialog.addMessage("2. Clear outside selection (set to black)");
Dialog.addMessage("3. Full image preserved (no crop)");

Dialog.addMessage("Click OK to continue or Cancel to skip");
Dialog.show();"""

            macro += f"""

// save full TL for overlay image (before ROI masking)
run("Select None");
run("Duplicate...", "duplicate channels=" + tl_channel);
rename("TL_raw");
run("Grays");

// duplicate full composite for ROI masking
selectWindow(title);
run("Select None");
setBackgroundColor(0, 0, 0);
setOption("BlackBackground", true);
run("Duplicate...", "duplicate");
roi_image = getTitle();

// apply ROI mask on the duplicate
nROIs = roiManager("count");
if (nROIs > 1) {{
    indices = newArray(nROIs);
    for (ri = 0; ri < nROIs; ri++) indices[ri] = ri;
    roiManager("Select", indices);
    roiManager("Combine");
    run("Clear Outside", "stack");
}} else if (nROIs == 1) {{
    roiManager("Select", 0);
    run("Clear Outside", "stack");
}}
run("Select None");
print("  ROI applied - cleared outside selection (set to black)");

// keep full image dimensions (no crop)
run("Select None");

// close the original, keep the ROI-processed duplicate
selectWindow(title);
close();
selectWindow(roi_image);
title = roi_image;"""

            if self.debug_mode:
                macro += f"""
Dialog.create("DEBUG - ROI Applied");
Dialog.addMessage("ROI has been applied:");
Dialog.addMessage("1. Duplicated image with selection");
Dialog.addMessage("2. Cleared outside the selection (set to black)");
Dialog.addMessage("3. Full image preserved (no crop)");

Dialog.addMessage("This preserves your freehand shape while removing excess area.");

Dialog.addMessage("Click OK to continue or Cancel to skip");
Dialog.show();"""
            
        else:
            # if not, use whole image analysis (no ROI selection)
            macro += f"""
// Using whole image for analysis (no ROI selection)
print("  Analyzing whole image - ROI selection disabled");

// Get image dimensions for recording
getDimensions(width, height, channels, slices, frames);
roi_x = 0;
roi_y = 0;
roi_width = width;
roi_height = height;
getStatistics(roi_area_actual);

print("  Image area: " + roi_area_actual + " pixels");

// save full TL for overlay image
run("Select None");
run("Duplicate...", "duplicate channels=" + tl_channel);
rename("TL_raw");
run("Grays");
selectWindow(title);"""

            if self.debug_mode:
                macro += f"""
Dialog.create("DEBUG - Whole Image Mode");
Dialog.addMessage("Analyzing whole image (no ROI).");
Dialog.addMessage("Image area: " + roi_area_actual + " pixels");

Dialog.addMessage("Click OK to continue with analysis");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// ANALYSIS WORKFLOW

print("  Starting analysis workflow...");

// STEP 5: Split channels (second time, for analysis)
print("  Splitting channels for analysis...");
run("Split Channels");"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Channels Split for Analysis");
Dialog.addMessage("Channels have been split again for analysis.");

Dialog.addMessage("Based on your earlier assignment:");
Dialog.addMessage("- Protein ({protein}): C" + protein_channel);
Dialog.addMessage("- Tomato Lectin: C" + tl_channel);
Dialog.addMessage("- Dextran: C" + dextran_channel);
Dialog.addMessage("- DAPI: C" + dapi_channel);

Dialog.addMessage("Click OK to continue with TL thresholding");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""
        
        macro += f"""

// Process TL channel first
// Try to select the TL channel window - it should be named C[n]-[title]
tl_window_name = tl_channel_prefix + title;
if (!isOpen(tl_window_name)) {{
    // If exact name doesn't work, try to find it by pattern
    print("  Looking for TL channel window...");
    list = getList("image.titles");
    for (i = 0; i < list.length; i++) {{
        if (indexOf(list[i], tl_channel_prefix) >= 0) {{
            tl_window_name = list[i];
            print("  Found TL channel: " + tl_window_name);
            break;
        }}
    }}
}}
selectWindow(tl_window_name);
rename("TL");

// Find and rename Protein channel NOW (needed for bg correction later)
protein_window_name = protein_channel_prefix + title;
if (!isOpen(protein_window_name)) {{
    print("  Looking for Protein channel window...");
    list = getList("image.titles");
    for (i = 0; i < list.length; i++) {{
        if (indexOf(list[i], protein_channel_prefix) >= 0) {{
            protein_window_name = list[i];
            print("  Found Protein channel: " + protein_window_name);
            break;
        }}
    }}
}}
selectWindow(protein_window_name);
rename("Protein");
print("  Renamed channels: TL and Protein");

// Create image subfolder for this image
img_save_dir = "{images_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}";
File.makeDirectory(img_save_dir);
print("  Image subfolder: " + img_save_dir);

// SAVE CHANNEL MIPs (before any processing)

// Save Protein MIP
selectWindow("Protein");
run("Duplicate...", " ");
run("Enhance Contrast", "saturated=0.35");
run("RGB Color");
run("Scale Bar...", "width={scale_bar_size} height=15 font=40 color=White background=None location=[Lower Right] bold");
saveAs("Tiff", img_save_dir + "/{protein}_MIP.tif");
close();
print("  Saved: {protein} MIP");

// Save TL MIP
selectWindow("TL");
run("Duplicate...", " ");
run("Enhance Contrast", "saturated=0.35");
run("RGB Color");
run("Scale Bar...", "width={scale_bar_size} height=15 font=40 color=White background=None location=[Lower Right] bold");
saveAs("Tiff", img_save_dir + "/TL_MIP.tif");
close();
print("  Saved: TL MIP");

// Switch back to TL for processing
selectWindow("TL");

selectWindow("TL");"""



        # ILASTIK TL: compare methods and let user choose
        if ilastik_tl_path:
            binary_path = ilastik_tl_path.replace('_TL_dominant.tif', '_TL_binary.tif')
            prob_path_tl = ilastik_tl_path.replace('_TL_dominant.tif', '_TL_prob.tif')
            dominant_path = ilastik_tl_path

            if ilastik_tl_prompt:
                macro += f"""

// ILASTIK TL: Open all variants for comparison
print("  Loading ilastik TL variants for comparison...");
binary_path = "{binary_path}";
prob_path = "{prob_path_tl}";
dominant_path = "{dominant_path}";

selectWindow("TL");
rename("Original TL");

if (File.exists(binary_path)) {{ open(binary_path); rename("Binary mask"); run("Green"); }}
if (File.exists(prob_path)) {{ open(prob_path); rename("Prob weighted"); run("Green"); }}
if (File.exists(dominant_path)) {{ open(dominant_path); rename("Dominant-class"); run("Green"); }}

selectWindow("Original TL");
run("Green");

choices = newArray("Original TL", "Binary mask", "Prob weighted", "Dominant-class");
for (ci = 0; ci < choices.length; ci++) {{
    if (isOpen(choices[ci])) {{
        selectWindow(choices[ci]);
        run("Enhance Contrast", "saturated=0.35");
    }}
}}

if (isOpen("Protein")) {{
    selectWindow("Protein");
    setLocation(-2000, -2000);
}}

sw = screenWidth;
sh = screenHeight - 80;
hw = floor(sw / 2);
hh = floor(sh / 2);
if (isOpen("Original TL")) {{ selectWindow("Original TL"); setLocation(0, 0, hw, hh); }}
if (isOpen("Binary mask")) {{ selectWindow("Binary mask"); setLocation(hw, 0, hw, hh); }}
if (isOpen("Prob weighted")) {{ selectWindow("Prob weighted"); setLocation(0, hh, hw, hh); }}
if (isOpen("Dominant-class")) {{ selectWindow("Dominant-class"); setLocation(hw, hh, hw, hh); }}

Dialog.create("Select TL Method");
Dialog.addMessage("Compare the 4 TL options and select one for analysis.");
Dialog.addMessage("Original = raw TL MIP (no ilastik)");
Dialog.addMessage("Binary = ilastik vessel mask applied to TL");
Dialog.addMessage("Prob weighted = TL x vessel probability");
Dialog.addMessage("Dominant-class = per-slice weighted projection");
Dialog.addChoice("Use:", choices, "Dominant-class");
Dialog.show();
tl_choice = Dialog.getChoice();
print("  User selected: " + tl_choice);

for (ci = 0; ci < choices.length; ci++) {{
    if (choices[ci] != tl_choice && isOpen(choices[ci])) {{
        selectWindow(choices[ci]);
        close();
    }}
}}

selectWindow(tl_choice);
rename("TL");
run("Maximize");
run("Green");
run("Enhance Contrast", "saturated=0.35");

if (isOpen("Protein")) {{
    selectWindow("Protein");
    setLocation(0, 0);
}}
selectWindow("TL");
print("  Using " + tl_choice + " as TL channel");

// close any leftover comparison windows
list = getList("image.titles");
for (i = list.length - 1; i >= 0; i--) {{
    win = list[i];
    if (win != "TL" && win != "Protein" && win != "TL_raw" && win != title) {{
        selectWindow(win);
        close();
    }}
}}
"""
            else:
                macro += f"""

dominant_path = "{dominant_path}";
if (File.exists(dominant_path)) {{
    selectWindow("TL");
    rename("Original TL (reference)");

    open(dominant_path);
    rename("TL");

    selectWindow("Original TL (reference)");
    run("Green");
    run("Enhance Contrast", "saturated=0.35");
    selectWindow("TL");
    run("Green");
    run("Enhance Contrast", "saturated=0.35");

    if (isOpen("Protein")) {{
        selectWindow("Protein");
        setLocation(-2000, -2000);
    }}

    sw = screenWidth;
    sh = screenHeight - 80;
    selectWindow("Original TL (reference)");
    setLocation(0, 0, floor(sw/2), sh);
    selectWindow("TL");
    setLocation(floor(sw/2), 0, floor(sw/2), sh);
    waitForUser("ilastik TL loaded",
        "Left: Original TL (reference)\\n" +
        "Right: Dominant-class weighted (will be used)\\n\\n" +
        "Click OK to proceed with the dominant-class version.\\n" +
        "Or close it and rename 'Original TL (reference)' to 'TL' to use raw.");

    if (isOpen("Original TL (reference)")) {{
        selectWindow("Original TL (reference)");
        close();
    }}

    if (isOpen("Protein")) {{
        selectWindow("Protein");
        setLocation(0, 0);
    }}
    selectWindow("TL");
    run("Maximize");
    run("Enhance Contrast", "saturated=0.35");
    print("  Using dominant-class weighted TL");
}} else {{
    print("  WARNING: ilastik TL not found, using raw TL");
}}
"""

        # TL VESSEL MASK: either load saved or threshold fresh

        if self.reuse_rois:
            # REUSE MODE: load saved vessel mask, then re-apply current cleanup settings
            macro += f"""

// REUSE MODE: Load saved vessel mask
vessel_mask_path = "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}_vessel_mask.tif";
if (File.exists(vessel_mask_path)) {{
    print("  Loading saved vessel mask: " + vessel_mask_path);
    selectWindow("TL");
    close();
    open(vessel_mask_path);
    rename("TL");
    print("  Vessel mask loaded from previous run");
}} else {{
    print("  WARNING: No saved vessel mask found, falling back to auto-threshold");
    setAutoThreshold("{self.auto_threshold_method} dark");
    run("Convert to Mask");
    getStatistics(area, tl_mean);
    if (tl_mean > 127) {{
        run("Invert");
    }}
}}

// RE-FILTER: apply current TL_REMOVE_SMALL setting
// close/fill are baked into the saved mask from the original run
tl_min_size = {TL_REMOVE_SMALL};
if (tl_min_size > 0) {{
    print("  Removing small particles (<" + tl_min_size + " px)...");
    run("Analyze Particles...", "size=" + tl_min_size + "-Infinity show=Masks");
    mask_title = getTitle();
    selectWindow("TL");
    close();
    selectWindow(mask_title);
    rename("TL");
    
    // check for inverted mask
    getStatistics(area, tl_mean);
    if (tl_mean > 127) {{
        run("Invert");
        print("  WARNING: mask inverted after filtering - auto-corrected");
    }}
}}
print("  Reuse filter complete");
"""

        else:
            # FRESH MODE: threshold + cleanup + save
            macro += f"""

// PRE-THRESHOLD: Gaussian blur to bridge gaps in patchy TL staining
tl_blur = {TL_GAUSSIAN_BLUR};
if (tl_blur > 0) {{
    print("  Applying Gaussian blur (sigma=" + tl_blur + ") to smooth patchy TL...");
    run("Gaussian Blur...", "sigma=" + tl_blur);
}}

// load previous vessel mask as overlay reference (if exists)
prev_mask_path = "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}_vessel_mask.tif";
if (File.exists(prev_mask_path)) {{
    open(prev_mask_path);
    rename("prev_mask_temp");
    setThreshold(128, 255);
    run("Create Selection");
    resetThreshold();
    // transfer as non-destructive overlay on TL
    selectWindow("TL");
    run("Restore Selection");
    run("Add Selection...", "stroke=yellow width=1");
    run("Select None");
    selectWindow("prev_mask_temp");
    close();
    selectWindow("TL");
    print("  Previous vessel mask shown as yellow overlay");
}}

// remove reference overlay before thresholding
run("Remove Overlay");

// re-apply ROI mask if ilastik replaced TL with full-image file
nROIs = roiManager("count");
if (nROIs > 0) {{
    selectWindow("TL");
    setBackgroundColor(0, 0, 0);
    if (nROIs > 1) {{
        indices = newArray(nROIs);
        for (ri = 0; ri < nROIs; ri++) indices[ri] = ri;
        roiManager("Select", indices);
        roiManager("Combine");
    }} else {{
        roiManager("Select", 0);
    }}
    run("Clear Outside");
    run("Select None");
    print("  ROI mask re-applied to TL channel");
}}

// STEP 6: Threshold TL channel
print("  Thresholding TL channel...");"""

            if self.auto_mode:
                macro += f"""
// [AUTO MODE] Apply {self.auto_threshold_method} threshold automatically
setAutoThreshold("{self.auto_threshold_method} dark");
run("Convert to Mask");
print("  [AUTO] Applied {self.auto_threshold_method} threshold to TL channel");
"""
            else:
                macro += f"""
setAutoThreshold("Default dark");
run("Threshold...");
waitForUser("Manual Threshold - TL Channel", 
    "The Threshold dialog is now open for the TL (vessel) channel.\\n\\n" +
    "Instructions:\\n" +
    "1. Adjust the sliders to highlight vessels\\n" +
    "2. Click 'Apply' in the threshold dialog\\n" +
    "3. The image will become binary (black and white)\\n" +
    "4. Vessels should be WHITE, background BLACK\\n" +
    "5. Then click OK in this message to continue");

// Ensure clean binary mask
run("Convert to Mask");
"""

            macro += f"""
// Check for inverted mask: if mean > 127, more pixels are white than black
// which likely means background is white and vessels are black (inverted)
getStatistics(area, tl_mean);
if (tl_mean > 127) {{
    run("Invert");
    print("  WARNING: TL mask appeared inverted (mean=" + tl_mean + ") - auto-corrected");
    print("           (vessels should be white, background black)");
}}

// POST-THRESHOLD: Morphological cleanup to fill gaps in vessel mask
"""

            macro += f"""
tl_close_iter = {TL_CLOSE_ITERATIONS};
if (tl_close_iter > 0) {{
    print("  Closing gaps in vessel mask (" + tl_close_iter + " iterations)...");
    for (ci = 0; ci < tl_close_iter; ci++) {{
        run("Close-");
    }}
}}

tl_fill = {str(TL_FILL_HOLES).lower()};
if (tl_fill) {{
    print("  Filling internal holes in vessel cross-sections...");
    run("Fill Holes");
}}

tl_min_size = {TL_REMOVE_SMALL};
if (tl_min_size > 0) {{
    print("  Removing small particles (<" + tl_min_size + " px)...");
    run("Analyze Particles...", "size=" + tl_min_size + "-Infinity show=Masks");
    // Analyze Particles creates a new "Mask of TL" window
    mask_title = getTitle();
    // Close original TL and rename mask
    selectWindow("TL");
    close();
    selectWindow(mask_title);
    rename("TL");
}}
"""

            macro += f"""
// Show cleaned mask for verification
getStatistics(area, tl_clean_mean);
print("  TL mask cleanup complete (mean=" + d2s(tl_clean_mean, 1) + ")");

// Save REUSABLE binary vessel mask to ROIs folder (0/255, no scale bar)
selectWindow("TL");
run("Duplicate...", " ");
saveAs("Tiff", "{roi_dir}/{mouse_type}_{brain_region}_{sample_number}_{protein}_vessel_mask.tif");
close();
selectWindow("TL");
print("  Saved reusable vessel mask to ROIs folder");
"""

        # Save display vessel mask image (RGB with scale bar) - both paths
        macro += f"""
// Save TL vessel mask display image (RGB with scale bar)
selectWindow("TL");
run("Duplicate...", " ");
run("RGB Color");
run("Scale Bar...", "width={scale_bar_size} height=15 font=40 color=White background=None location=[Lower Right] bold");
saveAs("Tiff", img_save_dir + "/TL_vessel_mask.tif");
close();
selectWindow("TL");
print("  Saved: TL vessel mask display image");"""

        if not self.auto_mode and not self.reuse_rois:
            macro += f"""
// Let user verify the cleaned TL mask
waitForUser("Verify TL Vessel Mask", 
    "The TL vessel mask has been cleaned up:\\n\\n" +
    "Applied: Gaussian blur (sigma={TL_GAUSSIAN_BLUR})\\n" +
    "         Close ({TL_CLOSE_ITERATIONS} iterations)\\n" +
    "         Fill Holes: {'Yes' if TL_FILL_HOLES else 'No'}\\n" +
    "         Remove small particles (<{TL_REMOVE_SMALL} px)\\n\\n" +
    "Check that:\\n" +
    "- Vessel outlines are continuous (gaps filled)\\n" +
    "- No excessive expansion beyond vessel boundaries\\n" +
    "- Small noise specks are removed\\n\\n" +
    "Click OK to continue with measurement.");
"""
        else:
            macro += f"""
print("  Skipping TL mask verification");
"""

        macro += f"""
// STEP 7: Measure vessel area fraction
run("Set Measurements...", "area_fraction");
run("Measure");
vessel_area = getResult("%Area", nResults-1);
print("  Vessel area fraction: " + vessel_area + "%");"""

        # ADAPTIVE BACKGROUND CORRECTION
        # Measures protein autofluorescence in off-vessel regions
        if self.bg_correction:
            macro += f"""

// ADAPTIVE BACKGROUND: Measure off-vessel autofluorescence
// Same approach as fragment analysis pipeline
print("  Measuring off-vessel background for adaptive threshold...");

// TL is currently 0/255 binary. Duplicate and invert to get off-vessel mask.
selectWindow("TL");
run("Duplicate...", " ");
rename("BG_mask");
run("Invert");  // Now: off-vessel = 255, vessels = 0

// Create selection from the inverted mask (selects off-vessel region)
setThreshold(128, 255);
run("Create Selection");
resetThreshold();

// Check we got a valid selection
if (selectionType() == -1) {{
    print("  WARNING: Could not create off-vessel selection - using full image for background");
    selectWindow("Protein");
    run("Set Measurements...", "mean median standard min max");
    run("Measure");
    bg_median = getResult("Median", nResults-1);
    bg_mean = getResult("Mean", nResults-1);
    bg_std = getResult("StdDev", nResults-1);
    protein_max = getResult("Max", nResults-1);
}} else {{
    // Transfer the off-vessel selection to the Protein channel
    selectWindow("Protein");
    run("Restore Selection");
    run("Set Measurements...", "mean median standard");
    run("Measure");
    bg_median = getResult("Median", nResults-1);
    bg_mean = getResult("Mean", nResults-1);
    bg_std = getResult("StdDev", nResults-1);
    run("Select None");
    
    // Measure full protein max for dynamic range
    run("Set Measurements...", "min max");
    run("Measure");
    protein_max = getResult("Max", nResults-1);
}}

// Compute adaptive threshold: median + fraction of dynamic range
// Median is more robust than mean (not skewed by bright debris or dark nuclei)
bg_fraction = {self.bg_fraction};
bg_threshold = bg_median + (bg_fraction * (protein_max - bg_median));

print("  Off-vessel background: median=" + d2s(bg_median, 2) + ", mean=" + d2s(bg_mean, 2) + ", SD=" + d2s(bg_std, 2));
print("  Protein max: " + d2s(protein_max, 1));
print("  Adaptive threshold: " + d2s(bg_threshold, 2) + " (median + " + d2s(bg_fraction*100, 0) + "% of range)");

// Clean up background mask
selectWindow("BG_mask");
close();
selectWindow("TL");
"""

        macro += f"""

// Save a copy of TL binary (0/255) for dextran vessel subtraction later
selectWindow("TL");
run("Duplicate...", " ");
rename("TL_binary");
selectWindow("TL");

// CRITICAL FIX: Normalize binary mask from 0/255 to 0/1
// Without this, Protein x TL gives values 255x too high
run("32-bit");
run("Divide...", "value=255");
print("  TL mask normalized to 0/1 for accurate multiplication");"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Vessel Area");
Dialog.addMessage("Vessel area fraction measured:");
Dialog.addMessage(vessel_area + "%");

Dialog.addMessage("Click OK to continue");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// Process protein channel (already renamed earlier)
selectWindow("Protein");"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Protein Channel");
Dialog.addMessage("Selected protein channel ({protein}).");

Dialog.addMessage("Click OK to create overlay");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// re-apply ROI mask to all analysis channels
nROIs = roiManager("count");
if (nROIs > 0) {{
    setBackgroundColor(0, 0, 0);
    mask_windows = newArray("TL", "Protein", "TL_binary");
    for (mw = 0; mw < mask_windows.length; mw++) {{
        if (isOpen(mask_windows[mw])) {{
            selectWindow(mask_windows[mw]);
            if (nROIs > 1) {{
                indices = newArray(nROIs);
                for (ri = 0; ri < nROIs; ri++) indices[ri] = ri;
                roiManager("Select", indices);
                roiManager("Combine");
            }} else {{
                roiManager("Select", 0);
            }}
            run("Clear Outside");
            run("Select None");
        }}
    }}
    print("  ROI mask applied to TL, Protein, and TL_binary");
}}

// STEP 8: Multiply protein x TL (32-bit result)
print("  Creating protein-vessel overlay...");

// Check that both required windows exist
if (!isOpen("Protein")) {{
    print("  ERROR: Protein window not found!");
}}
if (!isOpen("TL")) {{
    print("  ERROR: TL window not found!");
}}

// List all open windows for debugging
list = getList("image.titles");
print("  Open windows before multiplication:");
for (i = 0; i < list.length; i++) {{
    print("    - " + list[i]);
}}

// Create the overlay
imageCalculator("Multiply create 32-bit", "Protein", "TL");

// Check if a new window was created
list = getList("image.titles");
print("  Open windows after multiplication:");
for (i = 0; i < list.length; i++) {{
    print("    - " + list[i]);
}}

// The new window should be named "Result of Protein" or similar
// Find and rename it to "Overlay"
result_found = false;
for (i = 0; i < list.length; i++) {{
    if (indexOf(list[i], "Result") >= 0) {{
        selectWindow(list[i]);
        rename("Overlay");
        result_found = true;
        print("  Renamed result window to Overlay");
        break;
    }}
}}

if (!result_found) {{
    // If no "Result" window, the newest window is likely our overlay
    selectWindow(list[list.length-1]);
    rename("Overlay");
    print("  Renamed newest window to Overlay");
}}"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Overlay Created");
Dialog.addMessage("Created 32-bit overlay of Protein x TL.");

Dialog.addMessage("Click OK to measure MFI");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// STEP 9: Record MFI of overlaid image
selectWindow("Overlay");
run("Set Measurements...", "mean");
run("Measure");
overlay_mfi = getResult("Mean", nResults-1);
print("  Overlay MFI: " + overlay_mfi);"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Overlay MFI");
Dialog.addMessage("MFI of overlay: " + overlay_mfi);

Dialog.addMessage("Click OK to continue");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// STEP 10: Optional save with scale bar
// Initialize save variable
save_image = 0;
"""
        
        if self.auto_save_images or self.auto_mode or self.reuse_rois:
            macro += f"""
// Auto-save is enabled
save_image = 1;
print("  Auto-save enabled - will save image");
"""
        else:
            macro += f"""
// Ask user if they want to save using a Dialog
Dialog.create("Save Processed Image");
Dialog.addMessage("Would you like to save this processed image with scale bar?");
Dialog.addChoice("Your choice:", newArray("Yes", "No"), "Yes");
Dialog.show();
user_choice = Dialog.getChoice();
if (user_choice == "Yes") {{
    save_image = 1;
    print("  User chose to save image");
}} else {{
    save_image = 0;
    print("  User chose not to save image");
}}
"""
        
        # overlay color config
        pr, pg, pb = OVERLAY_PROTEIN_COLOR
        vr, vg, vb = OVERLAY_VESSEL_COLOR
        v_hex = "#{:02X}{:02X}{:02X}".format(vr, vg, vb)

        macro += f"""
// Now check if we should save
if (save_image == 1) {{

    // LAYER 1: grayscale TL
    selectWindow("TL_raw");
    run("Duplicate...", " ");
    rename("save_tl");
    run("Grays");
    run("Enhance Contrast", "saturated=0.35");
    run("8-bit");
    run("RGB Color");

    // LAYER 2: protein signal on vessels (bg-subtracted)
    selectWindow("Overlay");
    run("Duplicate...", " ");
    rename("save_protein");
"""

        if self.bg_correction:
            macro += f"""
    changeValues(0.001, bg_threshold, 0);
"""

        macro += f"""
    run("Enhance Contrast", "saturated=0.35");
    run("8-bit");

    // colorize protein [{pr},{pg},{pb}]
    selectWindow("save_protein");
    run("Duplicate...", " "); rename("prot_R"); run("Multiply...", "value={pr/255.0}");
    selectWindow("save_protein");
    run("Duplicate...", " "); rename("prot_G"); run("Multiply...", "value={pg/255.0}");
    selectWindow("save_protein");
    run("Duplicate...", " "); rename("prot_B"); run("Multiply...", "value={pb/255.0}");
    selectWindow("save_protein"); close();

    run("Merge Channels...", "c1=prot_R c2=prot_G c3=prot_B create");
    run("RGB Color");
    rename("protein_rgb");

    // composite: replace grayscale with colored protein where protein exists
    // create mask of protein pixels (matching bg correction)
    selectWindow("Overlay");
    run("Duplicate...", " ");
    rename("prot_mask");
    mask_thresh = bg_threshold;
    if (mask_thresh < 0.001) mask_thresh = 0.001;
    setThreshold(mask_thresh, 1e30);
    run("Convert to Mask");
    run("RGB Color");

    // zero out grayscale where protein signal exists
    imageCalculator("Subtract", "save_tl", "prot_mask");
    selectWindow("prot_mask"); close();

    // add colored protein on top (now visible against black)
    imageCalculator("Add create", "save_tl", "protein_rgb");
    save_window = getTitle();
    selectWindow("save_tl"); close();
    selectWindow("protein_rgb"); close();
    print("  Composite: grayscale TL + protein [{pr},{pg},{pb}]");

    // LAYER 3: vessel outline
    show_outline = {str(SAVE_VESSEL_OUTLINE).lower()};
    if (show_outline && isOpen("TL_binary")) {{
        selectWindow("TL_binary");
        setThreshold(128, 255);
        run("Create Selection");
        resetThreshold();

        selectWindow(save_window);
        run("Restore Selection");
        setForegroundColor({vr}, {vg}, {vb});
        run("Properties...", "stroke={v_hex} width=1");
        run("Flatten");
        flat_title = getTitle();
        selectWindow(save_window);
        close();
        selectWindow(flat_title);
        save_window = flat_title;
        print("  Added vessel outline [{vr},{vg},{vb}]");
    }}

    // scale bar and save
    selectWindow(save_window);
    run("Scale Bar...", "width={scale_bar_size} height=15 font=40 color=White background=None location=[Lower Right] bold");
    saveAs("Tiff", img_save_dir + "/{protein}_TL_overlay.tif");
    print("  Saved composite overlay");

    close();
    selectWindow("Overlay");
}}

// cleanup stale windows before overlay threshold
list = getList("image.titles");
for (i = list.length - 1; i >= 0; i--) {{
    win = list[i];
    if (win != "TL" && win != "TL_binary" && win != "TL_raw" && win != "Protein" && win != "Overlay") {{
        if (indexOf(win, "Dextran") < 0 && indexOf(win, "C") != 0) {{
            selectWindow(win);
            close();
        }}
    }}
}}

// STEP 11: Manual threshold for overlaid image
// Make sure we have the Overlay window
if (!isOpen("Overlay")) {{
    // If Overlay doesn't exist, try to find a window with overlay in name
    list = getList("image.titles");
    overlay_found = false;
    for (i = 0; i < list.length; i++) {{
        if (indexOf(list[i], "verlay") >= 0 || indexOf(list[i], "VERLAY") >= 0) {{
            selectWindow(list[i]);
            rename("Overlay");
            overlay_found = true;
            print("  Found and renamed overlay window: " + list[i]);
            break;
        }}
    }}
    if (!overlay_found) {{
        print("  ERROR: Could not find Overlay window for thresholding!");
        print("  Open windows:");
        for (i = 0; i < list.length; i++) {{
            print("    - " + list[i]);
        }}
    }}
}} else {{
    selectWindow("Overlay");
}}

print("  Thresholding overlay...");"""

        if self.bg_correction:
            # Use adaptive threshold from off-vessel background measurement
            macro += f"""
// ADAPTIVE BACKGROUND THRESHOLD
// Using off-vessel autofluorescence: threshold = bg_median + {int(self.bg_fraction*100)}% of range
print("  Applying adaptive background threshold: " + d2s(bg_threshold, 2));
setThreshold(bg_threshold, 1e30);
run("Convert to Mask");
print("  [BG CORRECTED] Only protein signal above background autofluorescence is counted");
"""
        elif self.auto_mode:
            macro += f"""
// [AUTO MODE] Apply {self.auto_threshold_method} threshold automatically to overlay
setAutoThreshold("{self.auto_threshold_method} dark");
run("Convert to Mask");
print("  [AUTO] Applied {self.auto_threshold_method} threshold to overlay");
"""
        else:
            macro += f"""
setAutoThreshold("Default dark");
run("Threshold...");
waitForUser("Manual Threshold - Overlay", 
    "The Threshold dialog is now open for the overlay image.\\n\\n" +
    "Instructions:\\n" +
    "1. Adjust sliders to highlight protein on vessels\\n" +
    "2. Click 'Apply' in the threshold dialog\\n" +
    "3. The image will become binary\\n" +
    "4. Protein signal should be WHITE, background BLACK\\n" +
    "5. Then click OK in this message to continue");

// Ensure clean binary
run("Convert to Mask");
"""

        macro += f"""
// Check for inverted mask on overlay too
getStatistics(area, overlay_thresh_mean);
if (overlay_thresh_mean > 127) {{
    run("Invert");
    print("  WARNING: Overlay mask appeared inverted - auto-corrected");
}}

// STEP 12: Measure protein coverage area fraction
run("Set Measurements...", "area_fraction");
run("Measure");
protein_coverage = getResult("%Area", nResults-1);
print("  Protein coverage: " + protein_coverage + "%");"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Protein Coverage");
Dialog.addMessage("Protein coverage area: " + protein_coverage + "%");

Dialog.addMessage("Click OK to calculate final metric");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// STEP 13: Calculate % protein coverage on vessels
percent_on_vessels = 0;
if (vessel_area > 0) {{
    percent_on_vessels = (protein_coverage / vessel_area) * 100;
}} else {{
    percent_on_vessels = 0;
}}
print("  % Protein coverage on vessels: " + percent_on_vessels + "%");

// Sanity check: coverage on vessels should not exceed 100%
if (percent_on_vessels > 100) {{
    print("  *** WARNING: Coverage on vessels exceeds 100% ("+d2s(percent_on_vessels,1)+"%) ***");
    print("  *** This suggests a mask issue - check TL threshold direction ***");
    print("  *** vessel_area=%Area=" + d2s(vessel_area,2) + "%, protein_coverage=%Area=" + d2s(protein_coverage,2) + "% ***");
}}"""

        if self.debug_mode:
            macro += f"""
Dialog.create("DEBUG - Final Calculation");
Dialog.addMessage("% Protein coverage on vessels:");
Dialog.addMessage(percent_on_vessels + "%");

Dialog.addMessage("(protein coverage / vessel area * 100)");

Dialog.addMessage("Click OK to finish");
Dialog.addMessage("Click Cancel to skip this image");
Dialog.show();"""

        macro += f"""

// close analysis windows no longer needed
if (isOpen("Overlay")) {{ selectWindow("Overlay"); close(); }}

// Process dextran if present
dextran_coverage = 0;
dextran_total = 0;
if (dextran_channel > 0) {{
    // Try to select the dextran channel window
    dextran_window_name = dextran_channel_prefix + title;
    if (!isOpen(dextran_window_name)) {{
        // If exact name doesn't work, try to find it by pattern
        print("  Looking for Dextran channel window...");
        list = getList("image.titles");
        for (i = 0; i < list.length; i++) {{
            if (indexOf(list[i], dextran_channel_prefix) >= 0) {{
                dextran_window_name = list[i];
                print("  Found Dextran channel: " + dextran_window_name);
                break;
            }}
        }}
    }}
    if (isOpen(dextran_window_name)) {{
        selectWindow(dextran_window_name);
        rename("Dextran");"""
    
        if self.debug_mode:
            macro += f"""
    waitForUser("DEBUG - Dextran Channel", 
        "Selected Dextran channel.\\nClick OK to threshold dextran.");"""
    
        macro += f"""
    
    // Threshold for dextran
    print("  Thresholding dextran...");"""

        if self.reuse_rois and saved_dex_threshold > 0:
            macro += f"""
    // REUSE: apply saved dextran threshold
    dextran_threshold = {saved_dex_threshold};
    setThreshold(dextran_threshold, 65535);
    run("Convert to Mask");
    print("  [REUSE] Applied saved dextran threshold: " + dextran_threshold);
"""
        elif self.auto_mode:
            macro += f"""
    // [AUTO MODE] Apply {self.auto_threshold_method} threshold automatically to dextran
    setAutoThreshold("{self.auto_threshold_method} dark");
    getThreshold(dextran_threshold, dex_upper);
    run("Convert to Mask");
    print("  [AUTO] Applied {self.auto_threshold_method} threshold to dextran (value=" + dextran_threshold + ")");
"""
        else:
            macro += f"""
    setAutoThreshold("Default dark");
    run("Threshold...");
    waitForUser("Manual Threshold - Dextran", 
        "The Threshold dialog is now open for the dextran channel.\\n\\n" +
        "Instructions:\\n" +
        "1. Adjust sliders to highlight dextran\\n" +
        "2. Click 'Apply' in the threshold dialog\\n" +
        "3. The image will become binary\\n" +
        "4. Then click OK in this message to continue");
    
    // capture threshold before converting
    getThreshold(dextran_threshold, dex_upper);
    print("  Dextran threshold captured: " + dextran_threshold);
    
    // Ensure clean binary
    run("Convert to Mask");
"""

        macro += f"""
    // Check for inverted mask
    getStatistics(area, dex_mean);
    if (dex_mean > 127) {{
        run("Invert");
        print("  WARNING: Dextran mask appeared inverted - auto-corrected");
    }}
    
    // Measure TOTAL dextran coverage (including intravascular)
    run("Set Measurements...", "area_fraction");
    run("Measure");
    dextran_total = getResult("%Area", nResults-1);
    print("  Total dextran coverage: " + dextran_total + "%");
    
    // SUBTRACT VESSEL MASK to get PARENCHYMAL dextran only
    // (dextran that has leaked out of vessels into surrounding tissue)
    if (isOpen("TL_binary")) {{
        print("  Subtracting vessel mask from dextran...");
        imageCalculator("Subtract", "Dextran", "TL_binary");
        
        // Measure parenchymal dextran (outside vessels only)
        run("Set Measurements...", "area_fraction");
        run("Measure");
        dextran_coverage = getResult("%Area", nResults-1);
        print("  Parenchymal dextran coverage: " + dextran_coverage + "%");
        print("  (intravascular dextran removed: " + d2s(dextran_total - dextran_coverage, 2) + "%)");
    }} else {{
        print("  WARNING: TL_binary not found - reporting total dextran (includes intravascular)");
        dextran_coverage = dextran_total;
    }}
    
    // Save dextran composite: grayscale TL + colored dextran + vessel outline
    dr = {OVERLAY_DEXTRAN_COLOR[0]};
    dg = {OVERLAY_DEXTRAN_COLOR[1]};
    db = {OVERLAY_DEXTRAN_COLOR[2]};
    
    selectWindow("TL_raw");
    run("Duplicate...", " ");
    rename("dex_tl");
    run("Grays");
    run("Enhance Contrast", "saturated=0.35");
    run("8-bit");
    run("RGB Color");
    
    // colorize dextran signal
    selectWindow("Dextran");
    run("Duplicate...", " ");
    rename("dex_signal");
    run("8-bit");
    selectWindow("dex_signal");
    run("Duplicate...", " "); rename("dR"); run("Multiply...", "value=" + d2s(dr/255.0, 4));
    selectWindow("dex_signal");
    run("Duplicate...", " "); rename("dG"); run("Multiply...", "value=" + d2s(dg/255.0, 4));
    selectWindow("dex_signal");
    run("Duplicate...", " "); rename("dB"); run("Multiply...", "value=" + d2s(db/255.0, 4));
    selectWindow("dex_signal"); close();
    run("Merge Channels...", "c1=dR c2=dG c3=dB create");
    run("RGB Color");
    rename("dex_rgb");
    
    // create mask and composite
    selectWindow("Dextran");
    run("Duplicate...", " ");
    rename("dex_mask");
    setThreshold(128, 255);
    run("Convert to Mask");
    run("RGB Color");
    imageCalculator("Subtract", "dex_tl", "dex_mask");
    selectWindow("dex_mask"); close();
    imageCalculator("Add create", "dex_tl", "dex_rgb");
    dex_save = getTitle();
    selectWindow("dex_tl"); close();
    selectWindow("dex_rgb"); close();
    
    // vessel outline
    show_outline = {str(SAVE_VESSEL_OUTLINE).lower()};
    if (show_outline && isOpen("TL_binary")) {{
        selectWindow("TL_binary");
        setThreshold(128, 255);
        run("Create Selection");
        resetThreshold();
        selectWindow(dex_save);
        run("Restore Selection");
        setForegroundColor({vr}, {vg}, {vb});
        run("Properties...", "stroke={v_hex} width=1");
        run("Flatten");
        flat_title = getTitle();
        selectWindow(dex_save);
        close();
        selectWindow(flat_title);
        dex_save = flat_title;
    }}
    
    selectWindow(dex_save);
    run("Scale Bar...", "width={scale_bar_size} height=15 font=40 color=White background=None location=[Lower Right] bold");
    saveAs("Tiff", img_save_dir + "/dextran_outside_vessels.tif");
    close();
    print("  Saved: dextran composite overlay");
"""
    
        if self.debug_mode:
            macro += f"""
    waitForUser("DEBUG - Dextran Coverage", 
        "Parenchymal dextran: " + dextran_coverage + "%\\nTotal dextran: " + dextran_total + "%\\nClick OK to save results.");"""
    
        macro += f"""
    }} else {{
        print("  Dextran channel not found or not available");
    }}
}}

// close Protein channel (analysis complete)
if (isOpen("Protein")) {{ selectWindow("Protein"); close(); }}

// Close DAPI channel if present (not used in analysis)
if (dapi_channel > 0) {{
    // Try to find and close the DAPI channel window
    dapi_window_name = dapi_channel_prefix + title;
    if (!isOpen(dapi_window_name)) {{
        // If exact name doesn't work, try to find it by pattern
        list = getList("image.titles");
        for (i = 0; i < list.length; i++) {{
            if (indexOf(list[i], dapi_channel_prefix) >= 0) {{
                dapi_window_name = list[i];
                break;
            }}
        }}
    }}
    if (isOpen(dapi_window_name)) {{
        selectWindow(dapi_window_name);
        close();
        print("  DAPI channel closed (not used in vessel analysis)");
    }}
}}
"""
        
        macro += f"""
// SAVE RESULTS

// Create results string
results = "vessel_area_percent:" + vessel_area + "\\n";
results = results + "mean_intensity:" + overlay_mfi + "\\n";
results = results + "protein_coverage_percent:" + protein_coverage + "\\n";
results = results + "percent_coverage_on_vessels:" + percent_on_vessels + "\\n";
results = results + "dextran_coverage_percent:" + dextran_coverage + "\\n";
results = results + "dextran_total_percent:" + dextran_total + "\\n";
results = results + "dextran_threshold:" + dextran_threshold + "\\n";
results = results + "bg_mean:" + bg_mean + "\\n";
results = results + "bg_median:" + bg_median + "\\n";
results = results + "bg_std:" + bg_std + "\\n";
results = results + "bg_threshold:" + bg_threshold + "\\n";
results = results + "roi_x:" + roi_x + "\\n";
results = results + "roi_y:" + roi_y + "\\n";
results = results + "roi_width:" + roi_width + "\\n";
results = results + "roi_height:" + roi_height + "\\n";
results = results + "roi_area:" + roi_area_actual + "\\n";
results = results + "original_width:" + original_width + "\\n";
results = results + "original_height:" + original_height + "\\n";
results = results + "meta_magnification:" + meta_magnification + "\\n";
results = results + "meta_pixel_size_um:" + meta_pixel_size_um + "\\n";
results = results + "meta_objective:" + meta_objective + "\\n";
results = results + "meta_tile_scan:" + meta_tile_scan + "\\n";
results = results + "meta_bit_depth:" + meta_bit_depth;"""

        macro += f"""
// Save to temp file
File.saveString(results, "{temp_dir}/{"" + blind_id if self.blinded else filename}_results.txt");

// Print summary
print("\\n  Results for {filename}:");
print("    Vessel area: " + d2s(vessel_area, 1) + "%");
print("    Overlay MFI: " + d2s(overlay_mfi, 1));
print("    Background: median=" + d2s(bg_median, 2) + " mean=" + d2s(bg_mean, 2) + " threshold=" + d2s(bg_threshold, 2));
print("    Protein coverage: " + d2s(protein_coverage, 1) + "%");
print("    % Coverage on vessels: " + d2s(percent_on_vessels, 1) + "%");
print("    Dextran (parenchymal): " + d2s(dextran_coverage, 1) + "%");
print("    Dextran (total): " + d2s(dextran_total, 1) + "%");"""

        if self.debug_mode:
            macro += f"""
waitForUser("DEBUG - Analysis Complete", 
    "Analysis complete for {filename}\\n\\n" +
    "Results:\\n" +
    "- Vessel area: " + d2s(vessel_area, 1) + "%\\n" +
    "- Overlay MFI: " + d2s(overlay_mfi, 1) + "\\n" +
    "- Protein coverage: " + d2s(protein_coverage, 1) + "%\\n" +
    "- % on vessels: " + d2s(percent_on_vessels, 1) + "%\\n" +
    "- Dextran (parenchymal): " + d2s(dextran_coverage, 1) + "%\\n" +
    "\\n" +
    "Click OK to close all windows and finish.");"""

        macro += f"""

// Clean up
setBatchMode(false);
if (isOpen("TL_binary")) {{ selectWindow("TL_binary"); close(); }}
run("Close All");
run("Clear Results");
roiManager("Reset");

print("\\n  [OK] Analysis complete for {filename}");

// Exit
run("Quit");
"""
        #finish macro
        return macro
    
    def process_all_images(self, image_files):
        
        # sort images
        sorted_images = self._sort_images(image_files)

        if self.blinded:
            import random
            if self.debug_mode:
                random.seed(42)
            random.shuffle(sorted_images)
            for idx, img in enumerate(sorted_images, 1):
                self.blind_map[img['filename']] = f"Image_{idx:03d}"
            print("[BLINDED] File order randomized, cohort names hidden")
        
        # if in debug mode, filter to only one image per type
        if self.debug_mode:
            print("\n[DEBUG MODE] 1 image per mouse type")
            filtered_images = []
            seen_mouse_types = set()
            
            for img_data in sorted_images:
                mouse_type = img_data['mouse_type']
                if mouse_type not in seen_mouse_types:
                    filtered_images.append(img_data)
                    seen_mouse_types.add(mouse_type)
                    print(f"  Selected for {mouse_type}: {img_data['filename']}")
            
            sorted_images = filtered_images
            print(f"  Reduced from {len(image_files)} to {len(sorted_images)} images\n")
        
        total = len(sorted_images)
        print("")
        print(f"PROCESSING {total} IMAGES WITH UPDATED WORKFLOW")
        if self.auto_mode:
            print(f"FULL AUTO MODE - {self.auto_threshold_method} threshold, no manual intervention")
        if self.debug_mode:
            print(f"DEBUG MODE ENABLED - Each step will pause for verification")
        if not self.use_roi:
            print(f"WHOLE IMAGE MODE - No ROI selection")
        if self.show_background_ui:
            print(f"MANUAL BACKGROUND - UI will be shown for background subtraction")
        print("")
        
        if not self.blinded:
            self._print_parsing_summary(sorted_images)
        
        # print("")
        
        all_results = []
        
        # process by group
        current_mouse_type = None
        current_region = None
        current_mouse_id = None
        
        for i, img_data in enumerate(sorted_images, 1):
            # print group headers
            if not self.blinded and img_data['mouse_type'] != current_mouse_type:
                current_mouse_type = img_data['mouse_type']
                print("")
                print(f"MOUSE TYPE: {current_mouse_type}")
                print("")
            
            if not self.blinded and img_data['brain_region'] != current_region:
                current_region = img_data['brain_region']
                print(f"\n--- Brain Region: {current_region} ---")
            
            if not self.blinded and img_data['mouse_id'] != current_mouse_id:
                current_mouse_id = img_data['mouse_id']
                print(f"\n  Mouse ID: {current_mouse_id}")
            
            results = self.process_single_image_with_roi(img_data, i, total)
            all_results.append(results)
            
            # save results progressively
            self._save_results(all_results)
            
            # save organized results
            self._save_organized_results(all_results)
            
            # brief pause between images
            if i < total:
                time.sleep(1)
        
        print("")
        print("ALL IMAGES PROCESSED")
        print("")
        
        # print unblinding key if blinded
        if self.blinded:
            print("\n" + "="*50)
            print("UNBLINDING KEY")
            print("")
            for fname, bid in sorted(self.blind_map.items(), key=lambda x: x[1]):
                print(f"  {bid} = {fname}")
            print("")

        # final summary (use full merged CSV if available)
        if self.results_csv.exists():
            full_results = pd.read_csv(self.results_csv).to_dict('records')
            print(f"Summary includes all {len(full_results)} images from previous + current runs")
            self._print_summary(full_results)
        else:
            self._print_summary(all_results)
        
        return pd.DataFrame(all_results)
    
    def _print_parsing_summary(self, sorted_images):
            
        print("\nImage Parsing Summary:")
        print("")
        
        # Count by categories
        mouse_types = {}
        regions = {}
        proteins = {}
        mouse_ids = set()
        
        for img in sorted_images:
            mouse_types[img['mouse_type']] = mouse_types.get(img['mouse_type'], 0) + 1
            regions[img['brain_region']] = regions.get(img['brain_region'], 0) + 1
            proteins[img['protein']] = proteins.get(img['protein'], 0) + 1
            mouse_ids.add(img['mouse_id'])
        
        print(f"Total unique mice: {len(mouse_ids)}")
        print(f"Mouse types: {mouse_types}")
        print(f"Brain regions: {regions}")
        print(f"Proteins: {proteins}")
        print("")
    
    def _save_results(self, results_list):
            
        new_df = pd.DataFrame(results_list)
        
        # Load existing results if they exist
        if self.results_csv.exists():
            existing_df = pd.read_csv(self.results_csv)
            existing_df = existing_df[~existing_df['filename'].isin(new_df['filename'])]
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            combined_df = new_df
        
        combined_df.to_csv(self.results_csv, index=False)
        print(f"  >> Main results updated: {self.results_csv} ({len(combined_df)} total images)")


    def _save_organized_results(self, results_list):
            
        df = pd.DataFrame(results_list)
        
        # Save by mouse type (create dirs on-the-fly for unexpected types)
        for mouse_type in df['mouse_type'].unique():
            if mouse_type and mouse_type != 'unknown':
                mouse_dir = self.by_mouse_dir / mouse_type
                mouse_dir.mkdir(parents=True, exist_ok=True)
                mouse_df = df[df['mouse_type'] == mouse_type]
                mouse_file = mouse_dir / f"{mouse_type}_all_results.csv"
                mouse_df.to_csv(mouse_file, index=False)
                
                # Save by region within mouse type
                for region in mouse_df['brain_region'].unique():
                    if region and region != 'unknown':
                        region_dir = mouse_dir / region
                        region_dir.mkdir(parents=True, exist_ok=True)
                        region_df = mouse_df[mouse_df['brain_region'] == region]
                        region_file = region_dir / f"{mouse_type}_{region}_results.csv"
                        region_df.to_csv(region_file, index=False)
        
        # Save by brain region
        for region in df['brain_region'].unique():
            if region and region != 'unknown':
                region_dir = self.by_region_dir / region
                region_dir.mkdir(parents=True, exist_ok=True)
                region_df = df[df['brain_region'] == region]
                region_file = region_dir / f"{region}_all_results.csv"
                region_df.to_csv(region_file, index=False)
        
        # Save by protein
        for protein in df['protein'].unique():
            if protein and protein != 'unknown':
                protein_dir = self.by_protein_dir / protein
                protein_dir.mkdir(parents=True, exist_ok=True)
                protein_df = df[df['protein'] == protein]
                protein_file = protein_dir / f"{protein}_all_results.csv"
                protein_df.to_csv(protein_file, index=False)
        
        # Save by individual mouse
        for mouse_id in df['mouse_id'].unique():
            if mouse_id and mouse_id != 'unknown_unknown':
                mouse_df = df[df['mouse_id'] == mouse_id]
                mouse_file = self.output_dir / f"mouse_{mouse_id}_results.csv"
                mouse_df.to_csv(mouse_file, index=False)
    
    def _print_summary(self, results_list):
            
        df = pd.DataFrame(results_list)
        
        # filter out errors
        df_valid = df[~df['filename'].str.contains('error', na=False)]
        
        if not df_valid.empty and 'percent_coverage_on_vessels' in df_valid.columns:
            print("\nOVERALL SUMMARY:")
            print(f"  Successfully processed: {len(df_valid)}/{len(df)} images")
            
            # runs stats with SEM
            print(f"\n  Overall Statistics:")
            mean_val = df_valid['percent_coverage_on_vessels'].mean()
            sem_val = df_valid['percent_coverage_on_vessels'].sem()
            print(f"    % Protein on Vessels: {mean_val:.1f} Â± {sem_val:.1f}% (mean Â± SEM)")
            
            if 'vessel_area_percent' in df_valid:
                vessel_mean = df_valid['vessel_area_percent'].mean()
                vessel_sem = df_valid['vessel_area_percent'].sem()
                print(f"    Vessel Area: {vessel_mean:.1f} Â± {vessel_sem:.1f}%")
            
            if 'mean_intensity' in df_valid:
                mfi_mean = df_valid['mean_intensity'].mean()
                mfi_sem = df_valid['mean_intensity'].sem()
                print(f"    MFI: {mfi_mean:.1f} Â± {mfi_sem:.1f}")
            
            # stats by mouse type
            print(f"\n  By Mouse Type:")
            for mouse_type in MOUSE_TYPES:
                if mouse_type in df_valid['mouse_type'].values:
                    type_df = df_valid[df_valid['mouse_type'] == mouse_type]
                    print(f"    {mouse_type}:")
                    print(f"      N = {len(type_df)} images from {len(type_df['mouse_id'].unique())} mice")
                    mean_val = type_df['percent_coverage_on_vessels'].mean()
                    sem_val = type_df['percent_coverage_on_vessels'].sem()
                    print(f"      % Protein on Vessels: {mean_val:.1f} Â± {sem_val:.1f}% (mean Â± SEM)")
            
            # Stats by brain region
            print(f"\n  By Brain Region:")
            for region in sorted(df_valid['brain_region'].unique()):
                if region != 'unknown':
                    region_df = df_valid[df_valid['brain_region'] == region]
                    print(f"    {region}:")
                    print(f"      N = {len(region_df)} images")
                    mean_val = region_df['percent_coverage_on_vessels'].mean()
                    sem_val = region_df['percent_coverage_on_vessels'].sem()
                    print(f"      % Protein on Vessels: {mean_val:.1f} Â± {sem_val:.1f}% (mean Â± SEM)")
            
            # Stats by individual mouse
            print(f"\n  By Individual Mouse:")
            for mouse_id in sorted(df_valid['mouse_id'].unique())[:5]:  # Show first 5 mice
                if mouse_id != 'unknown_unknown':
                    mouse_df = df_valid[df_valid['mouse_id'] == mouse_id]
                    print(f"    {mouse_id}:")
                    print(f"      Images: {len(mouse_df)}, Regions: {', '.join(mouse_df['brain_region'].unique())}")
                    print(f"      % Protein on Vessels: {mouse_df['percent_coverage_on_vessels'].mean():.1f}%")
        
        print(f"\nResults saved in organized structure:")
        print(f"  Main results: {self.results_csv}")
        print(f"  By mouse type: {self.by_mouse_dir}/")
        print(f"  By brain region: {self.by_region_dir}/")
        print(f"  By protein: {self.by_protein_dir}/")
        print(f"  Individual ROIs: {self.roi_dir}/")
        print(f"  Processed images: {self.images_dir}/")

# ilastik Pipeline
Run Step 1 once to export z-stacks, then Step 3 + Step 4 for each protein folder.
Skip this entire section if not using ilastik (ILASTIK_TL_MODE = 'off').

In [ ]:
# BACKUP TRAINING FILES (run before Step 1 re-exports)

backup_dir = os.path.join(TRAINING_DIR, "training_backups")
os.makedirs(backup_dir, exist_ok=True)

for fname in TRAINING_H5S:
    src = os.path.join(TRAINING_DIR, fname)
    dst = os.path.join(backup_dir, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  backed up: {fname}")

In [ ]:
# STEP 1: EXPORT TL Z-STACKS
# two passes: scan Z depths, then export padded to max Z

# pass 1: scan
print("Pass 1: scanning Z depths...")
file_info = []

for protein in PROTEINS:
    protein_dir = os.path.join(BASE_DIR, protein)
    if not os.path.exists(protein_dir):
        continue
    for fpath in sorted(glob.glob(os.path.join(protein_dir, '**', '*.czi'), recursive=True)):
        fname = os.path.basename(fpath)
        if not is_target(fname):
            continue
        stem = os.path.splitext(fname)[0]
        ch = get_channel_config_cached(fpath)
        zstack = extract_channel_zstack(fpath, ch['TL'])
        z_depth = zstack.shape[0] if zstack.ndim == 3 else 1
        print(f"  {protein}/{fname}: Z={z_depth}, TL=C{ch['TL']}, protein=C{ch['protein']}, dextran=C{ch['dextran']}, DAPI=C{ch['DAPI']}")
        file_info.append({'czi': fpath, 'stem': stem, 'protein': protein,
                          'z_depth': z_depth, 'ch': ch})

if not file_info:
    print("[!] No target files found.")
else:
    z_depths = [f['z_depth'] for f in file_info]
    TARGET_Z = max(z_depths)
    print(f"\nZ depths: min={min(z_depths)}, max={max(z_depths)}, padding to Z={TARGET_Z}")

    # save Z info
    with open(os.path.join(TRAINING_DIR, 'z_depth_info.json'), 'w') as f:
        json.dump({'target_z': TARGET_Z, 'files': {fi['stem']: fi['z_depth'] for fi in file_info}}, f, indent=2)

    # pass 2: export
    print(f"\nPass 2: exporting padded z-stacks (Z={TARGET_Z})...")
    for info in file_info:
        stem = info['stem']
        ch = info['ch']
        tl_out = os.path.join(TRAINING_DIR, f"{stem}_TL_zstack.h5")
        prot_out = os.path.join(TRAINING_DIR, f"{stem}_{info['protein']}_MIP.tif")
        tl_mip_out = os.path.join(TRAINING_DIR, f"{stem}_TL_MIP.tif")

        if RE_EXPORT or not os.path.exists(tl_out):
            print(f"  {stem} (Z={info['z_depth']} -> {TARGET_Z})")
            tl_zstack = extract_channel_zstack(info['czi'], ch['TL'])
            save_zstack_h5(pad_zstack(tl_zstack, TARGET_Z), tl_out)

            tl_mip = np.max(tl_zstack, axis=0) if tl_zstack.ndim == 3 else tl_zstack
            tifffile.imwrite(tl_mip_out, tl_mip.astype(np.uint16))

            if ch['protein'] > 0:
                prot_mip = extract_channel_mip(info['czi'], ch['protein'])
                tifffile.imwrite(prot_out, prot_mip.astype(np.uint16))
        else:
            print(f"  {stem}: already exported")

    print(f"\n{len(glob.glob(os.path.join(TRAINING_DIR, '*_TL_zstack.h5')))} h5 files in {TRAINING_DIR}")


In [ ]:
# STEP 3: HEADLESS PREDICTION

assert os.path.exists(ILP_PROJECT), f"Train ilastik first: {ILP_PROJECT}"
assert os.path.exists(PREDICT_DIR), f"Not found: {PREDICT_DIR}"

# find target .czi files
czi_files = sorted(glob.glob(os.path.join(PREDICT_DIR, '*.czi')))
target_czis = [f for f in czi_files if is_target(os.path.basename(f))]
print(f"{len(target_czis)} target images (of {len(czi_files)} total)")

# find corresponding h5s
tl_h5s = []
for fpath in target_czis:
    stem = os.path.splitext(os.path.basename(fpath))[0]
    tl_path = os.path.join(TRAINING_DIR, f"{stem}_TL_zstack.h5")
    if os.path.exists(tl_path):
        tl_h5s.append(tl_path)
    else:
        print(f"  [!] No z-stack for {stem} — run Step 1")

prob_dir = os.path.join(PREDICT_DIR, "ilastik_prob")
mask_dir = os.path.join(PREDICT_DIR, "vessel_masks")
os.makedirs(prob_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

# skip files that already have good outputs
if REANALYZE:
    tl_h5s_todo = tl_h5s
    print(f"REANALYZE=True: re-predicting all {len(tl_h5s_todo)} files")
else:
    tl_h5s_todo = []
    for h in tl_h5s:
        stem = os.path.splitext(os.path.basename(h))[0]
        prob_path = os.path.join(prob_dir, f"{stem}_prob.h5")
        if not os.path.exists(prob_path) or os.path.getsize(prob_path) < 1e6:
            tl_h5s_todo.append(h)
    print(f"{len(tl_h5s) - len(tl_h5s_todo)} already done, {len(tl_h5s_todo)} remaining")

if not tl_h5s_todo:
    print("All files predicted. Set REANALYZE=True to force re-run.")
else:
    output_template = os.path.join(prob_dir, '{nickname}_prob.h5')
    cmd = [
        ILASTIK_EXE, '--headless',
        f'--project={ILP_PROJECT}',
        '--export_source=Probabilities',
        f'--output_filename_format={output_template}',
        '--export_dtype=float32',
        '--output_format=compressed hdf5',
        '--readonly=1',
    ] + tl_h5s_todo

    print(f"Running ilastik headless (3D) on {len(tl_h5s_todo)} files...")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        line = line.strip()
        if 'Exporting to' in line:
            print(f"  {line.split('Exporting to ')[-1]}")
        elif 'Data shape' in line:
            print(f"  Shape: {line.split('Data shape: ')[-1]}")
        elif 'ERROR' in line:
            print(f"  [ERROR] {line}")
    process.wait()
    print(f"\nReturn code: {process.returncode}")

print(f"{len(glob.glob(os.path.join(prob_dir, '*.h5')))} probability files in {prob_dir}")

In [ ]:
# STEP 4: GENERATE WEIGHTED TL VARIANTS
# saves 3 versions per image for user selection in FIJI:
#   _TL_binary.tif     = TL MIP x binary vessel mask (ilastik segmentation)
#   _TL_prob.tif        = TL MIP x vessel probability (continuous weight)
#   _TL_dominant.tif    = TL MIP x dominant-class weighted projection (per-slice)

prob_dir = os.path.join(PREDICT_DIR, "ilastik_prob")
mask_dir = os.path.join(PREDICT_DIR, "vessel_masks")
os.makedirs(mask_dir, exist_ok=True)

# only process current targets
prob_files = []
for fpath in target_czis:
    stem = os.path.splitext(os.path.basename(fpath))[0]
    prob_path = os.path.join(prob_dir, f"{stem}_TL_zstack_prob.h5")
    if os.path.exists(prob_path) and os.path.getsize(prob_path) > 1e6:
        prob_files.append(prob_path)
    else:
        print(f"  [!] No prob file for {stem}")

print(f"{len(prob_files)} probability volumes to process")

results = []
for prob_path in prob_files:
    stem = os.path.basename(prob_path).replace('_TL_zstack_prob.h5', '').replace('_prob.h5', '')
    # load z-stack
    with h5py.File(os.path.join(TRAINING_DIR, f"{stem}_TL_zstack.h5"), 'r') as f:
        tl_zstack = f['data'][:].astype(np.float32)

    # load probabilities (with retry for file locks)
    prob = None
    for attempt in range(6):
        try:
            with h5py.File(prob_path, 'r') as f:
                prob = f['exported_data'][:]
            break
        except OSError:
            if attempt < 5:
                print(f"  {stem}: file locked, retrying in 10s... ({attempt+1}/6)")
                time.sleep(10)
            else:
                raise

    vp = prob[..., VESSEL_CHANNEL].astype(np.float32)
    mp = prob[..., MICROGLIA_CHANNEL].astype(np.float32)
    tl_mip = np.max(tl_zstack, axis=0)

    # binary: vessel is winning class at any z, applied to TL MIP
    vessel_mask = np.max(np.argmax(prob, axis=-1) == VESSEL_CHANNEL, axis=0).astype(np.float32)
    binary = tl_mip * vessel_mask

    # probability: TL MIP x max vessel confidence
    vessel_conf = np.max(vp, axis=0)
    prob_weighted = tl_mip * vessel_conf

    # dominant-class: per-slice, zero where microglia wins, weight by vessel prob
    micro_wins = (mp > vp).astype(np.float32)
    dominant = np.max(tl_zstack * vp * (1.0 - micro_wins), axis=0)

    # save all 3
    tifffile.imwrite(os.path.join(mask_dir, f"{stem}_TL_binary.tif"), binary.astype(np.uint16))
    tifffile.imwrite(os.path.join(mask_dir, f"{stem}_TL_prob.tif"), prob_weighted.astype(np.uint16))
    tifffile.imwrite(os.path.join(mask_dir, f"{stem}_TL_dominant.tif"), dominant.astype(np.uint16))

    print(f"  {stem}: binary/prob/dominant saved")
    results.append({'stem': stem})


print(f"\n{len(results)} image sets saved to {mask_dir}")

In [ ]:
# QC: COMPARE ALL ILASTIK METHODS
# shows original TL, binary mask, probability weighted, and dominant-class

prob_dir = os.path.join(PREDICT_DIR, "ilastik_prob")
mask_dir = os.path.join(PREDICT_DIR, "vessel_masks")

to_preview = []
for fpath in target_czis:
    stem = os.path.splitext(os.path.basename(fpath))[0]
    dpath = os.path.join(mask_dir, f"{stem}_TL_dominant.tif")
    if os.path.exists(dpath):
        to_preview.append(stem)

N = len(to_preview)
if N == 0:
    print("No processed images for current targets.")
else:
    fig, axes = plt.subplots(N, 4, figsize=(24, 6*N))
    if N == 1: axes = axes[np.newaxis, :]

    for row, stem in enumerate(to_preview):
        tl = tifffile.imread(os.path.join(TRAINING_DIR, f"{stem}_TL_MIP.tif")).astype(np.float32)
        binary = tifffile.imread(os.path.join(mask_dir, f"{stem}_TL_binary.tif")).astype(np.float32)
        prob_w = tifffile.imread(os.path.join(mask_dir, f"{stem}_TL_prob.tif")).astype(np.float32)
        dominant = tifffile.imread(os.path.join(mask_dir, f"{stem}_TL_dominant.tif")).astype(np.float32)

        axes[row, 0].imshow(make_green(norm(tl)))
        axes[row, 0].set_title(f'{stem}\nOriginal TL', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(make_green(norm(binary)))
        axes[row, 1].set_title('Binary mask', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(make_green(norm(prob_w)))
        axes[row, 2].set_title('Prob weighted', fontsize=9)
        axes[row, 2].axis('off')

        axes[row, 3].imshow(make_green(norm(dominant)))
        axes[row, 3].set_title('Dominant-class', fontsize=9)
        axes[row, 3].axis('off')

    plt.suptitle('ilastik Method Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(mask_dir, 'method_comparison_qc.png'), dpi=200, bbox_inches='tight')
    plt.show()

# ImageJ Analysis
Runs the v9 vessel analysis pipeline in FIJI.
Change PREDICT_DIR in config for each protein folder.

In [ ]:
# RUN ANALYSIS

def main():
    print("ANALYSIS")
    print("Expected format: (Protein)_TL_(Magnification)_(Mouse Type)_(Brain Region)_(n=#)")
    print("Example: HA_TL_20x_AD_MID_1")
    print("")
    # show the paths
    print(f"Fiji: {FIJI_PATH}")
    print(f"Input: {INPUT_DIR}")
    print(f"Output: {OUTPUT_DIR}")
    # show the mode choices
    print(f"DEBUG MODE: {'ENABLED' if DEBUG_MODE else 'DISABLED'}")
    print(f"ROI MODE: {'ENABLED - Manual ROI selection' if USE_ROI else 'DISABLED - Whole image analysis'}")
    print(f"BACKGROUND UI: {'MANUAL' if SHOW_BACKGROUND_UI else 'AUTOMATIC'}")
    print(f"AUTO SAVE: {'YES' if AUTO_SAVE_IMAGES else 'PROMPT EACH IMAGE'}")
    print(f"AUTO MODE: {'ENABLED (' + AUTO_THRESHOLD_METHOD + ' threshold)' if AUTO_MODE else 'DISABLED - Manual thresholds'}")
    print(f"BG CORRECTION: {'ENABLED (median + ' + str(int(BG_FRACTION*100)) + '% of dynamic range)' if BG_CORRECTION else 'DISABLED - Otsu/manual overlay threshold'}")
    print(f"ILASTIK TL: {ILASTIK_TL_MODE} (auto=AD/TBI/OLD use dominant-class, manual=user selects in FIJI, off=raw TL)")
    print(f"CHANNEL DETECTION: {'AUTO (from CZI metadata)' if AUTO_DETECT_CHANNELS else 'MANUAL (from CHANNEL_CONFIG)'}")
    if DEBUG_MODE:
        print("         (Processing only 1 image per mouse type)")
    print("")
    print("TL Vessel Mask Cleanup:")
    print(f"  Gaussian blur sigma: {TL_GAUSSIAN_BLUR} {'(disabled)' if TL_GAUSSIAN_BLUR == 0 else ''}")
    print(f"  Close iterations: {TL_CLOSE_ITERATIONS} {'(disabled)' if TL_CLOSE_ITERATIONS == 0 else ''}")
    print(f"  Fill holes: {TL_FILL_HOLES}")
    print(f"  Remove small particles: {TL_REMOVE_SMALL} px {'(disabled)' if TL_REMOVE_SMALL == 0 else ''}")
    print("")
    if AUTO_DETECT_CHANNELS:
        print("Channel Detection: AUTO (will read from CZI metadata per image)")
    else:
        print("Channel Configuration (fallback):")
        for mouse_type in MOUSE_TYPES:
            config = CHANNEL_CONFIG.get(mouse_type, {})
            print(f"  {mouse_type}: Protein=C{config.get('protein', '?')}, TL=C{config.get('TL', '?')}, ", end="")
            if config.get('dextran', 0) > 0:
                print(f"Dextran=C{config.get('dextran')}, ", end="")
            if config.get('DAPI', 0) > 0:
                print(f"DAPI=C{config.get('DAPI')}")
            else:
                print("")
    print("")
    
    # Initialize processor with all options
    processor = IndividualROIProcessor(
        FIJI_PATH, 
        OUTPUT_DIR, 
        debug_mode=DEBUG_MODE, 
        use_roi=USE_ROI,
        show_background_ui=SHOW_BACKGROUND_UI,
        auto_save_images=AUTO_SAVE_IMAGES,
        auto_mode=AUTO_MODE,
        auto_threshold_method=AUTO_THRESHOLD_METHOD,
        bg_correction=BG_CORRECTION,
        bg_fraction=BG_FRACTION,
        ilastik_tl_mode=ILASTIK_TL_MODE
    )
    
    # Find ALL images
    input_path = Path(INPUT_DIR)
    image_files = list(input_path.glob("*.czi"))
    
    if not image_files:
        print("No .czi files found, looking for .tif files...")
        image_files = list(input_path.glob("*.tif*"))
    
    if not image_files:
        print("\n[ERROR] No images found in input directory!")
        print(f"   Checked: {INPUT_DIR}")
        print("   Looking for: *.czi or *.tif files")
        return None
    
    print(f"\n Found {len(image_files)} images to process")
    
    # Check for existing results and ask about duplicates
    results_csv = Path(OUTPUT_DIR) / "vessel_analysis_results.csv"
    if results_csv.exists():
        existing_df = pd.read_csv(results_csv)
        existing_filenames = set(existing_df['filename'].dropna().values)
        
        # Check which images already have results
        incoming_filenames = {f.stem for f in image_files}
        duplicates = incoming_filenames & existing_filenames
        new_only = incoming_filenames - existing_filenames
        
        if duplicates:
            print(f"\n[!] Found {len(duplicates)} of {len(incoming_filenames)} images already in results CSV.")
            print(f"    ({len(existing_df)} total images in existing file)")
            print(f"    {len(new_only)} new images, {len(duplicates)} already processed.")
            
            # Count saved ROIs and vessel masks
            roi_folder = Path(OUTPUT_DIR) / "ROIs"
            saved_vessel_masks = list(roi_folder.glob("*_vessel_mask.tif")) if roi_folder.exists() else []
            
            print()
            print("    Options:")
            print("      1 = Process ALL (overwrite existing results for duplicates)")
            print("      2 = Skip existing (only process new images)")
            print("      3 = Cancel")
            print(f"      4 = Re-run ALL with saved masks ({len(saved_vessel_masks)} vessel masks)")
            print("          (useful for iterating BG offset, threshold settings, etc.)")
            choice = input("    Enter choice (1/2/3/4): ").strip()
            
            if choice == '3':
                print("Cancelled.")
                return None
            elif choice == '2':
                # Filter to only new images
                image_files = [f for f in image_files if f.stem not in existing_filenames]
                print(f"    Skipping {len(duplicates)} existing images. Processing {len(image_files)} new images.")
                if not image_files:
                    print("    No new images to process!")
                    return None
            elif choice == '4':
                if not saved_vessel_masks:
                    print("    [!] No saved vessel masks found in ROIs folder! Run analysis first.")
                    if roi_folder.exists():
                        all_files = list(roi_folder.iterdir())
                        print(f"    ROIs folder contains {len(all_files)} files:")
                        for f in all_files[:10]:
                            print(f"      {f.name}")
                        if len(all_files) > 10:
                            print(f"      ... and {len(all_files) - 10} more")
                    return None
                print(f"    Re-running all {len(incoming_filenames)} images with saved vessel masks.")
                processor.reuse_rois = True
                print("    No freehand ROIs found - using whole image mode.")
            else:
                print(f"    Processing all {len(incoming_filenames)} images (overwriting duplicates).")
        else:
            print(f"    Existing results: {len(existing_df)} images (no overlap with current batch)")
    
    print("\nStarting analysis...")
    print("Images will be sorted by: Mouse Type -> Brain Region -> Sample Number")
    
    if USE_ROI:
        print("Each image will open for FREEHAND ROI selection.")
    else:
        print("Whole image analysis mode - no ROI selection.")
    
    if DEBUG_MODE:
        print("\n[DEBUG MODE] Each processing step will pause for verification.")
        print("             Only 1 image per mouse type will be processed.")
    print("")
    
    # process images automatically
    results_df = processor.process_all_images(image_files)
    
    print("\n[COMPLETE]")
    print(f"All results saved and organized in: {OUTPUT_DIR}")
    
    # create summary plots using ALL results (including previous runs)
    import matplotlib.pyplot as plt
    
    # Load the full merged CSV (includes previous + current run)
    full_results_csv = processor.results_csv
    if full_results_csv.exists():
            all_results_df = pd.read_csv(full_results_csv)
            print(f"\nPlotting with {len(all_results_df)} total images (all runs combined)")
    else:
            all_results_df = results_df
    
    if all_results_df is not None and not all_results_df.empty:
            # Filter valid results
            valid_df = all_results_df[~all_results_df['filename'].str.contains('error', na=False)]
            
            if not valid_df.empty and 'percent_coverage_on_vessels' in valid_df.columns:
                # define the order and labels
                brain_regions = ['PFC', 'HIPP', 'MID']
                mouse_types_order = ['YOUNG', 'TBI', 'OLD', 'AD']
                mouse_labels = ['Control', 'TBI', 'Aged', 'AD']  # Display labels
                
                # Create figure with 2 subplots
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
                
                # plot 1: % protein Coverage on Vessels
                ax = ax1
                metric = 'percent_coverage_on_vessels'
                
                # set up bar positions
                x = np.arange(len(brain_regions))  # label locations
                width = 0.18  # width of bars
                
                # colors for each mouse type
                colors = ['red', 'green', 'blue', 'yellow']
                
                # plot bars for each mouse type
                for i, (mouse_type, label) in enumerate(zip(mouse_types_order, mouse_labels)):
                    means = []
                    sems = []
                    
                    for region in brain_regions:
                        subset = valid_df[(valid_df['mouse_type'] == mouse_type) & 
                                        (valid_df['brain_region'] == region)]
                        
                        if not subset.empty:
                            mean_val = subset[metric].mean()
                            sem_val = subset[metric].sem()
                            if pd.isna(sem_val):  # If only one sample
                                sem_val = 0
                            means.append(mean_val)
                            sems.append(sem_val)
                            
                            # Plot individual data points
                            n_points = len(subset)
                            if n_points > 0:
                                # Calculate position for dots
                                bar_pos = x[brain_regions.index(region)] + (i - 1.5) * width
                                # Add jitter to dots if multiple points
                                if n_points > 1:
                                    jitter = np.random.normal(0, width/8, n_points)
                                else:
                                    jitter = [0]
                                
                                ax.scatter(bar_pos + jitter, subset[metric], 
                                         color='black', s=20, alpha=0.6, zorder=3)
                        else:
                            means.append(0)
                            sems.append(0)
                    
                    # Plot bars with error bars
                    positions = x + (i - 1.5) * width
                    ax.bar(positions, means, width, yerr=sems,
                          label=label, color=colors[i], capsize=3, 
                          error_kw={'linewidth': 1.5}, alpha=0.8)
                
                # format plot 1
                ax.set_ylabel('% Protein Coverage on Vessels', fontsize=12)
                ax.set_title('Protein Coverage on Vessels', fontsize=14, fontweight='bold')
                ax.set_xticks(x)
                ax.set_xticklabels(brain_regions, fontsize=12)
                ax.legend(loc='upper right', fontsize=10)
                ax.set_ylim(0, ax.get_ylim()[1] * 1.1)  # Add 10% space at top
                ax.grid(axis='y', alpha=0.3)
                
                # Add "Global" group fro overall averages
                x_global = len(brain_regions) + 0.5
                ax.axvline(x=len(brain_regions) - 0.5, color='gray', linestyle='--', alpha=0.5)
                
                for i, (mouse_type, label) in enumerate(zip(mouse_types_order, mouse_labels)):
                    subset = valid_df[valid_df['mouse_type'] == mouse_type]
                    if not subset.empty:
                        mean_val = subset[metric].mean()
                        sem_val = subset[metric].sem()
                        if pd.isna(sem_val):
                            sem_val = 0
                        
                        bar_pos = x_global + (i - 1.5) * width
                        ax.bar(bar_pos, mean_val, width, yerr=sem_val,
                              color=colors[i], capsize=3, alpha=0.8)
                        
                        # Add individual points
                        n_points = len(subset)
                        if n_points > 1:
                            jitter = np.random.normal(0, width/8, n_points)
                        else:
                            jitter = [0] * n_points
                        ax.scatter(bar_pos + jitter, subset[metric], 
                                 color='black', s=20, alpha=0.6, zorder=3)
                
                # update x-axis to include Global
                ax.set_xticks(list(x) + [x_global])
                ax.set_xticklabels(brain_regions + ['Global'], fontsize=12)
                
                # plot 2: MFI
                if 'mean_intensity' in valid_df.columns:
                    ax = ax2
                    metric = 'mean_intensity'
                    
                    # set up bar positions
                    x = np.arange(len(brain_regions))
                    
                    # plot bars for each mouse type
                    for i, (mouse_type, label) in enumerate(zip(mouse_types_order, mouse_labels)):
                        means = []
                        sems = []
                        
                        for region in brain_regions:
                            subset = valid_df[(valid_df['mouse_type'] == mouse_type) & 
                                            (valid_df['brain_region'] == region)]
                            
                            if not subset.empty:
                                mean_val = subset[metric].mean()
                                sem_val = subset[metric].sem()
                                if pd.isna(sem_val):
                                    sem_val = 0
                                means.append(mean_val)
                                sems.append(sem_val)
                                
                                # Plot individual data points
                                n_points = len(subset)
                                if n_points > 0:
                                    bar_pos = x[brain_regions.index(region)] + (i - 1.5) * width
                                    if n_points > 1:
                                        jitter = np.random.normal(0, width/8, n_points)
                                    else:
                                        jitter = [0]
                                    
                                    ax.scatter(bar_pos + jitter, subset[metric], 
                                             color='black', s=20, alpha=0.6, zorder=3)
                            else:
                                means.append(0)
                                sems.append(0)
                        
                        # pplot bars with error bars
                        positions = x + (i - 1.5) * width
                        ax.bar(positions, means, width, yerr=sems,
                              label=label, color=colors[i], capsize=3, 
                              error_kw={'linewidth': 1.5}, alpha=0.8)
                    
                    # add Global group
                    ax.axvline(x=len(brain_regions) - 0.5, color='gray', linestyle='--', alpha=0.5)
                    
                    for i, (mouse_type, label) in enumerate(zip(mouse_types_order, mouse_labels)):
                        subset = valid_df[valid_df['mouse_type'] == mouse_type]
                        if not subset.empty:
                            mean_val = subset[metric].mean()
                            sem_val = subset[metric].sem()
                            if pd.isna(sem_val):
                                sem_val = 0
                            
                            bar_pos = x_global + (i - 1.5) * width
                            ax.bar(bar_pos, mean_val, width, yerr=sem_val,
                                  color=colors[i], capsize=3, alpha=0.8)
                            
                            # Add individual points
                            n_points = len(subset)
                            if n_points > 1:
                                jitter = np.random.normal(0, width/8, n_points)
                            else:
                                jitter = [0] * n_points
                            ax.scatter(bar_pos + jitter, subset[metric], 
                                     color='black', s=20, alpha=0.6, zorder=3)
                    
                    # format plot 2
                    ax.set_ylabel('Mean Fluorescence Intensity', fontsize=12)
                    ax.set_title('MFI of Protein-Vessel Overlay', fontsize=14, fontweight='bold')
                    ax.set_xticks(list(x) + [x_global])
                    ax.set_xticklabels(brain_regions + ['Global'], fontsize=12)
                    ax.legend(loc='upper right', fontsize=10)
                    ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
                    ax.grid(axis='y', alpha=0.3)
                
                # Overall figure adjustments
                plt.suptitle('Vessel Analysis Summary - Updated Workflow', fontsize=16, fontweight='bold', y=1.02)
                plt.tight_layout()
                
                # Save figure
                plot_file = processor.output_dir / "vessel_analysis_summary_updated.png"
                plt.savefig(plot_file, dpi=150, bbox_inches='tight')
                print(f"\nSummary plots saved: {plot_file}")
                plt.show()
                
    
    return results_df

results = main()

# Utilities
Optional cells for reviewing outputs.

In [ ]:
# SWEEP OVERLAY IMAGES INTO ONE FOLDER
# set OUTPUT_DIR here if not running from the top
# OUTPUT_DIR = r"path/to/output/directory"  # set here if running standalone

import os
import shutil
from pathlib import Path

REVIEW_DIR = os.path.join(OUTPUT_DIR, "overlay_review")
os.makedirs(REVIEW_DIR, exist_ok=True)

source_dir = Path(OUTPUT_DIR) / "processed_images"
overlays = sorted(source_dir.rglob("*_TL_overlay.tif"))

for f in overlays:
    dest_name = f"{f.parent.name}_{f.name}"
    shutil.copy2(f, os.path.join(REVIEW_DIR, dest_name))

print(f"{len(overlays)} overlays copied to {REVIEW_DIR}")